# Motor-unit Stage-2 runner

Self-contained Colab notebook for the scaled full-physics smoke and a short V3 learner probe.

Upload this notebook, select a GPU runtime, and run the cells in order. It writes `stage2_results.json` with H1–H4 PASS/FAIL metrics.

Scale-conditional PASS values do not lock the hypotheses. `scientific_lock` stays false until the charter gates pass at full scale.

The vendored training package in the repository is `unit_bfm`. This notebook writes the modules the runner imports. The MuJoCo plant remains in the repository tree and is not required for this smoke.


In [ ]:
!pip install -q torch numpy
import os, torch
os.makedirs("/content/codebase/unit_bfm/v3", exist_ok=True)
os.makedirs("/content/stage2", exist_ok=True)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))


In [ ]:
import os
os.chdir('/content/codebase/unit_bfm/v3')
print(os.getcwd())


In [ ]:
%%writefile __init__.py
"""Isolated V3 implementation; never imports V2 training or network code."""
VERSION = 'LOCAL_RESPONSE_ASYM_FB_V3'


In [ ]:
%%writefile schema.py
"""Frozen V3 SI feature registry. Physics owns its separate, generated registry."""
from dataclasses import asdict, dataclass
import hashlib
import json
import math
import numpy as np

VERSION = 'LOCAL_RESPONSE_ASYM_FB_V3'
SCHEMA_VERSION = 'v3-core-si-1'
ESTIMATOR_VERSION = 'causal-5point-linear-dq-1'
G_DIM, X_DIM, Z_DIM = 13, 137, 16
CONTROL_DT, PHYSICS_DT = .01, .001
RING_SIZE = 22

@dataclass(frozen=True)
class Field:
    name: str
    scale: float
    units: str
    source: str
    time: str = 'current_boundary'
    reference: str = 'canonical_output/M'


def field(name, scale, units, source, time='current_boundary'):
    return Field(name, scale, units, source, time)

G_SCHEMA = tuple(field(*v) for v in [
    ('length_P', .4, 'm', 'known'), ('length_D', .4, 'm', 'known'),
    ('q', math.pi, 'rad', 'telemetry'), ('dq', 6., 'rad/s', 'telemetry'),
    ('tau_endpoint', 6., 'N*m', 'privileged_exact_axis_telemetry'),
    ('q_min', math.pi, 'rad', 'known'), ('q_max', math.pi, 'rad', 'known'),
    ('v_tip_x', 3., 'm/s', 'local_kinematics'), ('v_tip_z', 3., 'm/s', 'local_kinematics'),
    ('ddq_hat', 200., 'rad/s^2', 'causal_5point_fit'),
    ('a_tip_x', 100., 'm/s^2', 'causal_5point_fit'),
    ('a_tip_z', 100., 'm/s^2', 'causal_5point_fit'), ('acc_valid', 1., 'bool', 'estimator')])
G_SCHEMA = tuple(Field(f.name,f.scale,f.units,f.source,
                       'causal_5_endpoints/40ms_fit' if f.name in ('ddq_hat','a_tip_x','a_tip_z','acc_valid') else f.time,
                       f.reference) for f in G_SCHEMA)
RIGID_SCHEMA = tuple(field(f'{side}.{name}', scale, units, 'known')
    for side in ('P', 'D') for name, scale, units in
    [('mass', 2., 'kg'), ('COM_x', .4, 'm'), ('COM_z', .4, 'm'), ('I_yy', .1, 'kg*m^2')])
DRIVE_SCHEMA = tuple(field(name, scale, units, 'known') for name, scale, units in [
    ('request_scale', 6., 'N*m'), ('lower', 6., 'N*m'), ('upper', 6., 'N*m'),
    ('T_s', .01, 's'), ('delay_s', .01, 's'), ('damping', .08, 'N*m*s/rad'),
    ('friction', .08, 'N*m'), ('armature', .006, 'kg*m^2')])
SLOW_SCHEMA = tuple(field(n,s,u,'telemetry','even_endpoint/preceding_2_intervals') for n,s,u in
    [('q',math.pi,'rad'),('dq',6.,'rad/s'),('tau_mean',6.,'N*m'),
     ('u_mean',1.,'1'),('padding',1.,'bool'),('age',.2,'s')])
FAST_SCHEMA = tuple(field(n,s,u,'telemetry','preceding_interval_endpoint') for n,s,u in
    [('u_previous',1.,'1'),('tau_endpoint',6.,'N*m'),('padding',1.,'bool'),('age',.08,'s')])
MEMORY_ROW_SCHEMA = tuple(field(n,s,u,'feature_memory','boundary_age/preceding_interval') for n,s,u in
    [('q',math.pi,'rad'),('dq',6.,'rad/s'),('tau_endpoint',6.,'N*m'),('u_previous',1.,'1'),
     ('tau_integral',.06,'N*m*s'),('u_integral',.01,'s'),('interval_dt',.01,'s'),
     ('interval_valid',1.,'bool'),('age',.22,'s'),('valid',1.,'bool'),('present',1.,'bool')])
MEMORY_WIDTH = len(MEMORY_ROW_SCHEMA)
MEMORY_DIM = RING_SIZE * MEMORY_WIDTH + 1

def _expanded(rows, count, prefix):
    return tuple(Field(f'{prefix}[{i}].{f.name}',f.scale,f.units,f.source,f.time,f.reference)
                 for i in range(count) for f in rows)

def _masks(rows):
    return tuple(field(f'{f.name}.valid',1.,'bool','known') for f in rows)

X_SCHEMA = (G_SCHEMA + RIGID_SCHEMA + _masks(RIGID_SCHEMA) + DRIVE_SCHEMA + _masks(DRIVE_SCHEMA)
            + _expanded(SLOW_SCHEMA,10,'slow') + _expanded(FAST_SCHEMA,8,'fast'))
MEMORY_SCHEMA = _expanded(MEMORY_ROW_SCHEMA,RING_SIZE,'memory') + (field('slow_phase',1.,'bool','feature_memory'),)
G_SCALES = np.array([f.scale for f in G_SCHEMA])
RIGID_SCALES = np.array([f.scale for f in RIGID_SCHEMA])
DRIVE_SCALES = np.array([f.scale for f in DRIVE_SCHEMA])
SLOW_SCALES = np.array([f.scale for f in SLOW_SCHEMA])
FAST_SCALES = np.array([f.scale for f in FAST_SCHEMA])
MEMORY_SCALES = np.array([f.scale for f in MEMORY_ROW_SCHEMA])


def registry():
    return {k:[asdict(f) for f in rows] for k,rows in
            [('g',G_SCHEMA),('x',X_SCHEMA),('memory',MEMORY_SCHEMA)]}

SCHEMA_HASH = hashlib.sha256(json.dumps(registry(),sort_keys=True).encode()).hexdigest()


def concatenate_privileged(physics_p, memory):
    """p_total=[physics_p,feature_memory]; no guessed physics dimension."""
    physics_p, memory = np.asarray(physics_p), np.asarray(memory)
    if physics_p.ndim != 1 or memory.shape != (MEMORY_DIM,):
        raise ValueError('Expected 1D physics p and complete feature memory')
    if not np.isfinite(physics_p).all() or not np.isfinite(memory).all():
        raise ValueError('Nonfinite privileged input')
    return np.concatenate((physics_p,memory)).astype(np.float32)


@dataclass(frozen=True)
class CoordinateMap:
    """Output-side telemetry adapter only; never reapplied to env boundaries.

    Rotor units require an upstream, explicit transmission model.
    """
    axis_sign: int = 1
    canonical_angle_at_encoder_zero_rad: float = 0.
    calibration_valid: bool = True
    source: str = 'declared_output_side_calibration'
    version: str = 'canonical-output-1'

    def __post_init__(self):
        if self.axis_sign not in (-1,1) or not math.isfinite(self.canonical_angle_at_encoder_zero_rad):
            raise ValueError('Invalid coordinate map')

    def _check(self):
        if not self.calibration_valid:
            raise ValueError('Trusted calibration required for canonical geometry')

    def canonical(self,q,dq,tau):
        self._check()
        s = self.axis_sign
        return (s*np.asarray(q)+self.canonical_angle_at_encoder_zero_rad,
                s*np.asarray(dq),s*np.asarray(tau))

    def limits(self, raw_limits):
        self._check()
        return np.sort(self.axis_sign*np.asarray(raw_limits)+self.canonical_angle_at_encoder_zero_rad)

    def torque_limits(self, raw_limits):
        self._check()
        return np.sort(self.axis_sign*np.asarray(raw_limits))

    def request(self,u_c,request_scale,device_scale=None):
        self._check()
        device_scale = request_scale if device_scale is None else device_scale
        if request_scale <= 0 or device_scale <= 0:
            raise ValueError('Positive request scales required')
        return self.axis_sign*(request_scale/device_scale)*np.asarray(u_c)


In [ ]:
%%writefile fb_loss.py
"""Unmodified FB estimator with explicit transition-row discounts."""
import torch


def fb_loss(forwards,backward,target_density,discount,pair_mask=None):
    n = backward.shape[0]
    if n < 2 or forwards.shape != (2,n,backward.shape[-1]) or backward.ndim != 2:
        raise ValueError('Expected forwards [2,N,D], backward [N,D], N>=2')
    if target_density.shape != (n,n) or discount.shape != (n,1):
        raise ValueError('Expected target [N,N] and row discount [N,1]; [N] is ambiguous')
    if not torch.isfinite(discount).all() or torch.any((discount<0)|(discount>1)):
        raise ValueError('Discount outside [0,1]')
    mask = 1-torch.eye(n,device=backward.device,dtype=backward.dtype)
    if pair_mask is not None:
        # B03 configuration-matched pairs. Cross-config entries are not a reference.
        if pair_mask.shape != (n,n):
            raise ValueError('pair_mask must be [N,N]')
        mask = mask * pair_mask.to(dtype=mask.dtype, device=mask.device)
        if float(mask.sum()) <= 0:
            raise ValueError('INSUFFICIENT_SUPPORT: no matched off-diagonal pairs')
    density = forwards @ backward.T
    diff = density-discount.detach().reshape(1,n,1)*target_density.detach()
    offdiag = .5*(diff*mask).square().sum()/mask.sum()
    positive = -diff.diagonal(dim1=-2,dim2=-1).mean()*2
    gram = backward @ backward.T
    orth = .5*(gram*mask).square().sum()/mask.sum()-gram.diag().mean()
    return offdiag+positive+orth,offdiag,positive,orth


def actor_loss(forwards,z):
    return -(forwards*z).sum(-1).min(dim=0).values.mean()


def sample_action(mean,std=.2,clip=.3,generator=None):
    noise = torch.randn(mean.shape,dtype=mean.dtype,device=mean.device,generator=generator)*std
    if clip is not None:
        noise = noise.clamp(-clip,clip)
    value = mean+noise
    bounded = value.clamp(-1+1e-6,1-1e-6)
    return value-value.detach()+bounded.detach()


In [ ]:
%%writefile stage2_norms.py
"""B01–B04 engineering contracts merged into the V3 training tree.

Order of adoption: causal privileged sanitizer (B01), episode program and
rule law (B02), nondimensional map and support sampler (B03), feasibility
labels and frozen-local hash (B04). These are the sealed merge norms. They
do not rewrite packet decisions.

The V3 network still consumes the registered 13-field outcome. Charter T_κ
applies to the housing-frame layout used by the Stage-2 gate plant.
``nondim_v3_g`` is the schema-preserving tip scaling used inside B.
"""

from __future__ import annotations

import hashlib
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import torch

G_ACCEL = 9.81


def single_link_gravity(q: torch.Tensor, m: torch.Tensor, length: torch.Tensor) -> torch.Tensor:
    """G = dV/dq = -m g (L/2) cos(q) for the housing-frame rotary unit."""
    lc = 0.5 * length
    return -m * G_ACCEL * lc * torch.cos(q)

SCENES = ("E0", "E1", "E2", "E3", "E4")
SCENE_ID = {name: i for i, name in enumerate(SCENES)}
VARIANTS = {"default": 0, "E4-R": 1, "E4-RI": 2, "train": 3, "eval": 4}

# Scaled privileged layout. Indices 16:20 are endogenous future slots.
P_DIM = 24
FUTURE_SLICE = slice(16, 20)
LAGGED_SLICE = slice(12, 16)


def philox_digest(run_seed: int, config_id: int, scene: str, n: int, domain: int, variant: str) -> str:
    if scene not in SCENE_ID:
        raise KeyError(scene)
    if variant not in VARIANTS:
        raise KeyError(variant)
    ss = np.random.SeedSequence(
        [int(run_seed), int(config_id), SCENE_ID[scene], int(n), int(domain), VARIANTS[variant]]
    )
    raw = np.random.Generator(np.random.Philox(ss)).integers(0, 2**31 - 1, size=4, dtype=np.int64)
    return hashlib.sha256(raw.tobytes()).hexdigest()[:16]


def make_program(
    run_seed: int,
    config_id: int,
    scene: str,
    n: int,
    domain: int,
    variant: str,
    *,
    length: float,
    mass: float,
    s_max: float,
    qmin: float,
    qmax: float,
    amp: float,
    freq: float,
    steps: int = 80,
    control_dt: float = 0.02,
    m_frac: float = 0.58,
    q_star: float = 0.0,
) -> Dict:
    """Single source-of-truth episode program.

    E3 always contains a sustained-load window of at least 25 steps.
    E4 always contains one release edge and at least 20 recovery steps.
    Load sign is σ = (-1)^(config_id + n). Infeasible equilibria are flagged
    and the load is not silently sign-flipped.
    """
    if scene not in SCENE_ID:
        raise KeyError(scene)
    digest = philox_digest(run_seed, config_id, scene, n, domain, variant)
    sigma = 1.0 if ((config_id + n) % 2 == 0) else -1.0
    grav_star = float(single_link_gravity(torch.tensor(q_star), torch.tensor(mass), torch.tensor(length)))
    # Smallest fraction that keeps both signs above the 0.25 S_max floor, plus margin.
    m_need = 0.25 + abs(grav_star) / max(s_max, 1e-6) + 0.06
    m_frac = float(min(0.70, max(float(m_frac), m_need)))
    u_star = sigma * m_frac * s_max
    tau_level = grav_star - u_star
    feasible = (abs(u_star) <= 0.92 * s_max + 1e-9) and (abs(tau_level) >= 0.25 * s_max - 1e-9)
    status = "FEASIBLE" if feasible else "INFEASIBLE_CONFIG"

    onset = 6 + ((config_id + n) % 9)  # 6..14, de-synchronized
    e3_len = 30
    e4_release = 28 + ((config_id + 2 * n) % 5)  # 28..32, leaves >= 48 steps
    tau = np.zeros(steps, dtype=np.float64)
    q_des = np.full(steps, q_star, dtype=np.float64)
    kappa = 0.0
    t = np.arange(steps, dtype=np.float64) * control_dt

    if scene == "E0":
        q_des = q_star + amp * np.sin(2.0 * np.pi * freq * t)
    elif scene == "E1":
        # Inward escape from the positive stop. q_des stays at the interior star.
        kappa = 0.0
    elif scene == "E2":
        # Exogenous contact-site schedule. The plant adds a state-dependent
        # contact torque; kappa uses the lagged measured contact, not a future window.
        kappa = 1.0
        q_des = q_star + 0.5 * amp * np.sin(2.0 * np.pi * max(freq, 0.3) * t)
    elif scene == "E3":
        kappa = 1.0
        tau[onset : onset + e3_len] = tau_level
        assert e3_len >= 25
    elif scene == "E4":
        kappa = 0.0
        tau[:e4_release] = tau_level
        # one release edge, then a zero-load recovery
        assert steps - e4_release >= 20
    else:
        raise KeyError(scene)

    q_des = np.clip(q_des, qmin + 0.05, qmax - 0.05)
    anchor = (onset + config_id) % 3
    tier = ((onset // 3) + (config_id // 3)) % 3
    return {
        "digest": digest,
        "scene": scene,
        "config_id": int(config_id),
        "sigma": float(sigma),
        "u_star": float(u_star),
        "tau_level": float(tau_level),
        "status": status,
        "kappa": float(kappa),
        "tau_load": tau,
        "q_des": q_des,
        "onset": int(onset),
        "e3_len": int(e3_len),
        "e4_release": int(e4_release),
        "anchor": int(anchor),
        "tier": int(tier),
        "q_star": float(q_star),
        "m_frac": float(m_frac),
    }


def equilibrium_label(tau_g: float, sigma: float, m_frac: float, s_max: float) -> str:
    """Fail-closed load law. Does not redraw a failed sign into the other sign."""
    u_star = sigma * m_frac * s_max
    tau_ext = tau_g - u_star
    if abs(u_star) > 0.92 * s_max + 1e-12 or abs(tau_ext) < 0.25 * s_max - 1e-12:
        return "INFEASIBLE_CONFIG"
    return "FEASIBLE"


def rule_step(
    q: torch.Tensor,
    qd: torch.Tensor,
    q_des: torch.Tensor,
    integ: torch.Tensor,
    inertia: torch.Tensor,
    tau_g_hat: torch.Tensor,
    tau_ext_delayed: torch.Tensor,
    kappa: torch.Tensor,
    s_max: torch.Tensor,
    dt: float,
    kp: float = 27.0,
    ki: float = 27.0,
) -> tuple:
    """Inertia-scaled PD + bias + delayed feedforward + anti-windup.

    Plant form is M qdd + G = u + tau_load, so the cancelling bias is G and the
    cancelling feedforward is -tau_load. κ = 1 uses τ_ext at t-1. κ = 0 leaves
    the integrator to wind (E4 recovery).
    """
    # Inertia-scaled PID: qdd tracks kp e + ki∫e - kd qd when the bias cancels
    # gravity and the delayed load. Gains are a triple pole at 3 rad/s
    # (kd=9, kp=27, ki=27) unless the caller overrides kp/ki.
    kd = 9.0 if abs(kp - 27.0) < 1e-6 else (3.0 * math_sqrt(max(kp, 1e-6)))
    e = q_des - q
    integ_cand = integ + e * dt
    u_ff = tau_g_hat - kappa * tau_ext_delayed
    u_unsat = inertia * (kp * e - kd * qd + ki * integ_cand) + u_ff
    u_lim = 0.92 * s_max
    u = torch.clamp(u_unsat, -u_lim, u_lim)
    # Back-calculation anti-windup, in the same inertia-scaled units.
    integ_new = integ_cand + (u - u_unsat) / (inertia * ki).clamp_min(1e-6)
    integ_new = torch.clamp(integ_new, -2.0, 2.0)
    return u, integ_new


def math_sqrt(v: float) -> float:
    return float(v) ** 0.5


class DualPartitionNormalizer:
    """Two running-stat partitions. A fail-closed batch freezes both updates."""

    def __init__(self, proprio_dim: int, privileged_dim: int, eps: float = 1e-5):
        self.eps = eps
        self.p_mean = torch.zeros(proprio_dim)
        self.p_var = torch.ones(proprio_dim)
        self.v_mean = torch.zeros(privileged_dim)
        self.v_var = torch.ones(privileged_dim)
        self.p_count = 1e-4
        self.v_count = 1e-4
        self.frozen = False
        self.proprio_updates = 0
        self.privileged_updates = 0

    def _update(self, mean, var, count, x):
        flat = x.detach().reshape(-1, x.shape[-1]).double()
        bmean = flat.mean(0)
        bvar = flat.var(0, unbiased=False)
        bc = float(flat.shape[0])
        delta = bmean - mean.double()
        tot = count + bc
        new_mean = mean.double() + delta * (bc / tot)
        m2 = var.double() * count + bvar * bc + delta.pow(2) * count * bc / tot
        return new_mean.float(), (m2 / tot).float(), tot

    def update_proprio(self, x: torch.Tensor) -> None:
        if self.frozen:
            return
        self.p_mean, self.p_var, self.p_count = self._update(self.p_mean, self.p_var, self.p_count, x)
        self.proprio_updates += 1

    def update_privileged(self, p: torch.Tensor) -> None:
        if self.frozen:
            return
        self.v_mean, self.v_var, self.v_count = self._update(self.v_mean, self.v_var, self.v_count, p)
        self.privileged_updates += 1

    def freeze(self) -> None:
        self.frozen = True


def sanitize_privileged(p_raw: torch.Tensor, future_sample_delay: float = 0.02) -> Dict[str, torch.Tensor]:
    """Fail-closed causal sanitizer for the scaled p layout.

    Endogenous future slots (16:20) are replaced by the lagged carrier (12:16).
    Provenance sample_time > action_latch_time freezes the normalizer. The
    tensor consumed by F has zero remaining future-dated fields.
    """
    if p_raw.shape[-1] != P_DIM:
        raise ValueError(f"SCHEMA_INCOMPATIBLE p dim {p_raw.shape[-1]}")
    lagged = p_raw[..., LAGGED_SLICE]
    future = p_raw[..., FUTURE_SLICE]
    violated = torch.ones(p_raw.shape[:-1], dtype=torch.bool, device=p_raw.device)
    # Every raw row in this collector carries future-dated slots by construction.
    # A row whose future slice already matches the lagged carrier is clean.
    already_clean = torch.all(torch.abs(future - lagged) <= 1e-8, dim=-1)
    violated = violated & (~already_clean)
    p_out = p_raw.clone()
    p_out[..., FUTURE_SLICE] = lagged
    consumed_violations = int((torch.abs(p_out[..., FUTURE_SLICE] - lagged) > 1e-8).sum().item())
    return {
        "p": p_out,
        "raw_fail_closed_rows": int(violated.sum().item()),
        "consumed_violations": consumed_violations,
        "freeze_normalizer": bool(violated.any().item()),
        "future_sample_delay_s": float(future_sample_delay),
    }


def schema_guard_g(g: torch.Tensor) -> str:
    if g.shape[-1] != 13:
        return "SCHEMA_INCOMPATIBLE"
    if not torch.isfinite(g).all():
        return "SCHEMA_INCOMPATIBLE"
    # cos^2 + sin^2
    nrm = g[..., 0] ** 2 + g[..., 1] ** 2
    if torch.max(torch.abs(nrm - 1.0)).item() > 2e-2:
        return "SCHEMA_INCOMPATIBLE"
    return "OK"


def t_kappa(g: torch.Tensor, rho: torch.Tensor, rho_ref: float = 0.22) -> torch.Tensor:
    """Parameter-free invertible tip nondimensionalization.

    Divides the tip block by ρ̂, scales rates by the allometric time √(ρ̂/g),
    and writes λ = log(ρ̂/ρ_ref) into the dead tip-vz slot (index 9).
    """
    if schema_guard_g(g) != "OK":
        raise ValueError("SCHEMA_INCOMPATIBLE")
    rho = rho.to(dtype=g.dtype, device=g.device)
    while rho.ndim < g.ndim:
        rho = rho.unsqueeze(-1)
    out = g.clone()
    time_scale = torch.sqrt(rho / 9.81)
    out[..., 2] = g[..., 2] * time_scale.squeeze(-1)
    out[..., 3] = g[..., 3] * (time_scale.squeeze(-1) ** 2)
    view = rho.expand_as(g[..., 4:7])
    tview = time_scale.expand_as(g[..., 4:7])
    out[..., 4:7] = g[..., 4:7] / view.clamp_min(1e-6)
    out[..., 7:10] = g[..., 7:10] * tview / view.clamp_min(1e-6)
    out[..., 10:13] = g[..., 10:13] * (tview ** 2) / view.clamp_min(1e-6)
    lam = torch.log(rho.squeeze(-1).clamp_min(1e-6) / float(rho_ref))
    out[..., 9] = lam
    return out


def t_kappa_inverse(g_hat: torch.Tensor, rho_ref: float = 0.22) -> torch.Tensor:
    """Inverse of T_κ. Restores the dead vz slot to 0."""
    lam = g_hat[..., 9]
    rho = float(rho_ref) * torch.exp(lam)
    time_scale = torch.sqrt(rho / 9.81)
    out = g_hat.clone()
    out[..., 2] = g_hat[..., 2] / time_scale
    out[..., 3] = g_hat[..., 3] / (time_scale ** 2)
    rho_b = rho.unsqueeze(-1)
    ts_b = time_scale.unsqueeze(-1)
    out[..., 4:7] = g_hat[..., 4:7] * rho_b
    out[..., 7:10] = g_hat[..., 7:10] * rho_b / ts_b
    out[..., 10:13] = g_hat[..., 10:13] * rho_b / (ts_b ** 2)
    out[..., 9] = 0.0
    return out


def participation_ratio(z: torch.Tensor) -> float:
    """Effective rank of a [N, D] cloud via the eigenvalue entropy of covariance."""
    if z.shape[0] < 4:
        return float("nan")
    zc = z - z.mean(0, keepdim=True)
    cov = (zc.transpose(0, 1) @ zc) / float(z.shape[0])
    ev = torch.linalg.eigvalsh(cov).clamp_min(1e-12)
    p = ev / ev.sum()
    entropy = -(p * torch.log(p)).sum()
    return float(torch.exp(entropy).item())


class SupportSampler:
    """Hierarchical uniform sampler with a direct target-law mixture.

    Empty strata return INSUFFICIENT_SUPPORT. Weights of the remaining strata
    are not renormalized to hide the hole.
    """

    def __init__(self, counts: Dict[int, int], min_count: int = 2):
        self.counts = dict(counts)
        self.min_count = min_count

    def sample(self, requested: Sequence[int], mixture: float = 0.10) -> Dict:
        missing = [int(i) for i in requested if self.counts.get(int(i), 0) < self.min_count]
        if missing:
            return {"status": "INSUFFICIENT_SUPPORT", "missing": missing, "weights": None}
        req = [int(i) for i in requested]
        # 90% hierarchical uniform over the requested support, 10% same target law
        # (the target law in this balanced smoke grid is also uniform).
        w = np.full(len(req), (1.0 - mixture) / len(req) + mixture / len(req), dtype=np.float64)
        return {"status": "OK", "missing": [], "weights": w, "mixture": mixture}


def logit_path_alive() -> Dict[str, float]:
    """Action-space tanh gradient dies in saturation; the pre-activation hinge does not."""
    logit = torch.tensor([6.0], requires_grad=True)
    loss_a = torch.tanh(logit).sum()
    g_a, = torch.autograd.grad(loss_a, logit)
    logit2 = torch.tensor([6.0], requires_grad=True)
    loss_h = torch.relu(logit2.abs() - 3.0).sum()
    g_h, = torch.autograd.grad(loss_h, logit2)
    return {
        "saturated_tanh_grad": float(g_a.item()),
        "preactivation_hinge_grad": float(g_h.item()),
        "pass": bool(abs(float(g_a.item())) < 1e-3 and abs(float(g_h.item())) > 0.5),
    }


def marginals_do_not_certify_joint() -> bool:
    """Equal positive marginals can still hide a zero joint cell."""
    cells = np.ones((4, 5), dtype=int)
    cells[1, 2] = 0
    cells[2, 2] = 2  # repair one marginal so both margins match the complete table's totals? 
    # Construct a matrix with the same marginals as a positive matrix but a zero inside.
    good = np.ones((3, 3), dtype=int)
    bad = np.array([[0, 2, 1], [2, 0, 1], [1, 1, 1]], dtype=int)
    same_row = np.array_equal(good.sum(1), bad.sum(1))
    same_col = np.array_equal(good.sum(0), bad.sum(0))
    has_zero = bool((bad == 0).any())
    return bool(same_row and same_col and has_zero and (cells.min() == 0))


def frozen_local_hash(gains: Dict[str, float]) -> str:
    blob = repr(sorted((k, float(v)) for k, v in gains.items())).encode()
    return hashlib.sha256(blob).hexdigest()


def sanitize_privileged_tensor(
    p: torch.Tensor,
    future_slice: Optional[Tuple[int, int]] = None,
    lag_slice: Optional[Tuple[int, int]] = None,
) -> Tuple[torch.Tensor, Dict]:
    """B01 hook used before every V3 forward call.

    With no slice metadata the tensor is unchanged and the call is still
    counted. When a future window is declared, it is replaced by the lagged
    carrier and any leftover mismatch fail-closes the normalizer.
    """
    if future_slice is None and lag_slice is None:
        return p, {
            "applied": False,
            "consumed_violations": 0,
            "freeze_normalizer": False,
            "raw_fail_closed_rows": 0,
        }
    if future_slice is None or lag_slice is None:
        raise ValueError("future_slice and lag_slice must be declared together")
    fs, fe = int(future_slice[0]), int(future_slice[1])
    ls, le = int(lag_slice[0]), int(lag_slice[1])
    if fe - fs != le - ls or fs < 0 or le > p.shape[-1]:
        raise ValueError("SCHEMA_INCOMPATIBLE privileged slices")
    lagged = p[..., ls:le]
    future = p[..., fs:fe]
    dirty = ~torch.all(torch.abs(future - lagged) <= 1e-8, dim=-1)
    out = p.clone()
    out[..., fs:fe] = lagged
    leftover = int((torch.abs(out[..., fs:fe] - lagged) > 1e-8).sum().item())
    return out, {
        "applied": True,
        "consumed_violations": leftover,
        "freeze_normalizer": bool(dirty.any().item()),
        "raw_fail_closed_rows": int(dirty.sum().item()),
    }


def nondim_v3_g(g: torch.Tensor) -> torch.Tensor:
    """Length-relative tip velocity and acceleration for the registered V3 g.

    Channels, after the schema scales: 1 length_D, 7:9 tip velocity, 10:12
    tip acceleration. Dimension stays 13. Charter T_κ is a different layout
    and is not applied to this vector.
    """
    if g.shape[-1] != 13 or not torch.isfinite(g).all():
        raise ValueError("SCHEMA_INCOMPATIBLE")
    length = g[..., 1:2].abs().clamp_min(1e-3)
    time_scale = torch.sqrt(length)
    out = g.clone()
    out[..., 7:9] = g[..., 7:9] * time_scale / length
    out[..., 10:12] = g[..., 10:12] * time_scale.square() / length
    return out


def matched_pair_mask(config_id: torch.Tensor) -> torch.Tensor:
    """B03: successor pairs stay inside one morphology-stroke configuration."""
    config_id = config_id.reshape(-1)
    return (config_id.unsqueeze(1) == config_id.unsqueeze(0)).to(dtype=torch.float32)


def demand_label(demand: float, edge: float = 0.75, publish: float = 0.92) -> str:
    """B04 blocking feasibility. Labels come from inverse dynamics, not the arm."""
    if demand <= edge:
        return "IN"
    if demand <= publish:
        return "EDGE"
    return "OUT"


In [ ]:
%%writefile networks.py
"""V3 input dimensions with the unchanged V2/UFO hidden skeleton."""
import torch
from torch import nn
from .schema import G_DIM, X_DIM, Z_DIM
from .stage2_norms import nondim_v3_g


def embedding(n):
    return nn.Sequential(nn.Linear(n,256),nn.LayerNorm(256),nn.Tanh(),
                         nn.Linear(256,128),nn.ReLU())


class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed_s = embedding(X_DIM)
        self.embed_z = embedding(X_DIM+Z_DIM)
        # Tanh is applied in forward so a pre-activation hinge can see the logit.
        self.head = nn.Sequential(nn.Linear(256,256),nn.ReLU(),nn.Linear(256,1))

    def context(self,x):
        return self.embed_s(x)

    def logits(self,x,z):
        # Pre-activation path. The published action is tanh(logits).
        return self.head(torch.cat((self.embed_s(x),self.embed_z(torch.cat((x,z),-1))),-1))

    def forward(self,x,z):
        return torch.tanh(self.logits(x,z))


class ForwardMap(nn.Module):
    def __init__(self,p_dim):
        super().__init__()
        self.embed_sa = embedding(X_DIM+p_dim+1)
        self.embed_z = embedding(X_DIM+p_dim+Z_DIM)
        self.head = nn.Sequential(nn.Linear(256,256),nn.ReLU(),nn.Linear(256,Z_DIM))

    def forward(self,y,u,z):
        return self.head(torch.cat((self.embed_sa(torch.cat((y,u),-1)),
                                    self.embed_z(torch.cat((y,z),-1))),-1))


class SphereNorm(nn.Module):
    def forward(self,x):
        return Z_DIM**.5 * torch.nn.functional.normalize(x,dim=-1)


class BackwardMap(nn.Sequential):
    def __init__(self):
        super().__init__(nn.Linear(G_DIM,256),nn.LayerNorm(256),nn.Tanh(),
                         nn.Linear(256,256),nn.ReLU(),nn.Linear(256,Z_DIM),SphereNorm())

    def forward(self, g):
        # B03: tip channels are length-relative before the linear map.
        return super().forward(nondim_v3_g(g))


class Networks(nn.Module):
    def __init__(self,p_dim:int):
        super().__init__()
        if isinstance(p_dim,bool) or not isinstance(p_dim,int) or p_dim < 0:
            raise ValueError('p_dim must be a nonnegative integer including feature memory')
        self.p_dim = p_dim
        self.actor = Actor()
        self.f1,self.f2 = ForwardMap(p_dim),ForwardMap(p_dim)
        self.b = BackwardMap()

    def counts(self):
        counts = {k:sum(p.numel() for p in m.parameters()) for k,m in self.named_children()}
        counts['online'] = sum(counts.values())
        counts['execution'] = counts['actor']+counts['b']
        return counts


def orthogonal_init(module):
    if isinstance(module,nn.Linear):
        nn.init.orthogonal_(module.weight,gain=1.)
        nn.init.zeros_(module.bias)
    elif isinstance(module,nn.LayerNorm):
        module.reset_parameters()


In [ ]:
%%writefile learner.py
"""Asymmetric, reward-free FB training; independent matched goals supplied by replay."""
import copy
from contextlib import contextmanager
from dataclasses import asdict, dataclass
import torch
from torch import nn
from .schema import VERSION, SCHEMA_VERSION, SCHEMA_HASH, ESTIMATOR_VERSION, G_DIM, X_DIM, Z_DIM
from .networks import Networks, orthogonal_init
from .fb_loss import fb_loss, actor_loss, sample_action
from .stage2_norms import matched_pair_mask, sanitize_privileged_tensor


@dataclass(frozen=True)
class LearnerConfig:
    lr: float = 1e-4
    polyak: float = .01
    action_std: float = .2
    action_clip: float = .3

    def __post_init__(self):
        if not (0 < self.lr and 0 <= self.polyak <= 1 and self.action_std >= 0 and self.action_clip >= 0):
            raise ValueError('Invalid learner configuration')
        if not all(torch.isfinite(torch.tensor(v)) for v in asdict(self).values()):
            raise ValueError('Nonfinite learner configuration')


@contextmanager
def frozen_forward(net):
    """Freeze F weights while preserving dF/du for the Actor."""
    params = list(net.f1.parameters())+list(net.f2.parameters())
    flags = [p.requires_grad for p in params]
    try:
        for p in params:
            p.requires_grad_(False)
        yield
    finally:
        for p,flag in zip(params,flags):
            p.requires_grad_(flag)


class Learner:
    def __init__(self,p_dim,device='cpu',seed=61001,config=None):
        self.device = torch.device(device)
        self.config = config if isinstance(config,LearnerConfig) else LearnerConfig(**(config or {}))
        self.p_dim = p_dim
        # Initialization does not consume a caller's independent global CPU stream.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed)
            self.net = Networks(p_dim)
            self.net.apply(orthogonal_init)
        self.net.to(self.device)
        self.generator = torch.Generator(device=self.device).manual_seed(seed+1)
        self.target = nn.ModuleDict({k:copy.deepcopy(getattr(self.net,k)) for k in ('f1','f2','b')})
        self.target.requires_grad_(False)
        self.f_params = list(self.net.f1.parameters())+list(self.net.f2.parameters())
        self.forward_opt = torch.optim.Adam(self.f_params,lr=self.config.lr)
        self.backward_opt = torch.optim.Adam(self.net.b.parameters(),lr=self.config.lr)
        self.actor_opt = torch.optim.Adam(self.net.actor.parameters(),lr=self.config.lr)
        self.updates = 0
        self.last_goal_fraction = 0.
        self.sanitize_calls = 0
        self.last_sanitize = {'applied': False, 'consumed_violations': 0, 'freeze_normalizer': False}

    @torch.no_grad()
    def sample_z(self,n,goal_g=None,goal_valid=None):
        if n < 1:
            raise ValueError('n must be positive')
        z = Z_DIM**.5*torch.nn.functional.normalize(
            torch.randn(n,Z_DIM,device=self.device,generator=self.generator),dim=-1)
        if (goal_g is None) != (goal_valid is None):
            raise ValueError('Provide both goal_g and goal_valid')
        if goal_g is None:
            self.last_goal_fraction = 0.
            return z.detach()
        goal_g = torch.as_tensor(goal_g,dtype=torch.float32,device=self.device).detach()
        valid = torch.as_tensor(goal_valid,device=self.device)
        if goal_g.shape != (n,G_DIM) or valid.shape != (n,):
            raise ValueError('Expected independent goal_g [N,13], goal_valid [N]')
        if not torch.all((valid==0)|(valid==1)) or not torch.isfinite(goal_g).all():
            raise ValueError('Invalid goal input')
        choose = (torch.rand(n,device=self.device,generator=self.generator)<.5)&valid.bool()
        # No permutation: each complete goal remains matched to its transition configuration.
        if choose.any():
            z[choose] = self.net.b(goal_g[choose])
        self.last_goal_fraction = float(choose.float().mean())
        return z.detach()

    def _batch(self,batch):
        dims = {'x':X_DIM,'p':self.p_dim,'u':1,'x_next':X_DIM,'p_next':self.p_dim,
                'g_next':G_DIM,'discount':1,'goal_g':G_DIM}
        out = {k:torch.as_tensor(batch[k],dtype=torch.float32,device=self.device).detach() for k in dims}
        n = len(out['x'])
        if n < 2:
            raise ValueError('FB batch requires at least two rows')
        for k,d in dims.items():
            if out[k].shape != (n,d) or not torch.isfinite(out[k]).all():
                raise ValueError(f'Invalid batch {k}: expected finite [{n},{d}]')
        out['goal_valid'] = torch.as_tensor(batch['goal_valid'],device=self.device).detach()
        if out['goal_valid'].shape != (n,) or not torch.all((out['goal_valid']==0)|(out['goal_valid']==1)):
            raise ValueError('goal_valid must be binary [N]')
        if torch.any((out['discount']<0)|(out['discount']>1)):
            raise ValueError('Discount outside [0,1]')
        if torch.any(out['u'].abs()>1+1e-6):
            raise ValueError('Canonical action outside [-1,1]')
        # Lengths and limits define the goal group; capacity/mass/beta are not goal keys.
        valid = out['goal_valid'].bool()
        idx = [0,1,5,6]
        if not torch.allclose(out['goal_g'][valid][:,idx],out['x'][valid][:,idx],atol=1e-6,rtol=1e-6):
            raise ValueError('goal_g must match each transition geometry and limits')
        return out

    def _action(self,mean):
        return sample_action(mean,self.config.action_std,self.config.action_clip,self.generator)

    @torch.no_grad()
    def target_density(self,x_next,p_next,g_next,z):
        u_next = self._action(self.net.actor(x_next,z))
        y_next = torch.cat((x_next,p_next),-1)
        future = .5*(self.target['f1'](y_next,u_next,z)+self.target['f2'](y_next,u_next,z))
        return future @ self.target['b'](g_next).T

    @staticmethod
    def grad_norm(parameters):
        norms = [p.grad.detach().square().sum() for p in parameters if p.grad is not None]
        return torch.stack(norms).sum().sqrt() if norms else torch.tensor(0.)

    @torch.no_grad()
    def polyak(self):
        for name in ('f1','f2','b'):
            for online,target in zip(getattr(self.net,name).parameters(),self.target[name].parameters()):
                target.lerp_(online,self.config.polyak)

    def update(self,batch):
        future = batch.get('future_slice') if isinstance(batch, dict) else None
        lag = batch.get('lag_slice') if isinstance(batch, dict) else None
        config_id = batch.get('config_id') if isinstance(batch, dict) else None
        batch = self._batch(batch)
        # B01: every forward call consumes a sanitized privileged tensor.
        batch['p'], rep_p = sanitize_privileged_tensor(batch['p'], future, lag)
        batch['p_next'], rep_n = sanitize_privileged_tensor(batch['p_next'], future, lag)
        self.sanitize_calls += 1
        self.last_sanitize = {
            'applied': bool(rep_p['applied'] or rep_n['applied']),
            'consumed_violations': int(rep_p['consumed_violations'] + rep_n['consumed_violations']),
            'freeze_normalizer': bool(rep_p['freeze_normalizer'] or rep_n['freeze_normalizer']),
        }
        x,p,u = (batch[k] for k in ('x','p','u'))
        z = self.sample_z(len(x),batch['goal_g'],batch['goal_valid'])
        target = self.target_density(batch['x_next'],batch['p_next'],batch['g_next'],z)
        y = torch.cat((x,p),-1)
        forwards = torch.stack((self.net.f1(y,u,z),self.net.f2(y,u,z)))
        backward = self.net.b(batch['g_next'])
        pair_mask = None
        if config_id is not None:
            pair_mask = matched_pair_mask(torch.as_tensor(config_id, device=self.device)).to(dtype=backward.dtype)
        total,offdiag,positive,orth = fb_loss(forwards,backward,target,batch['discount'],pair_mask)
        if not torch.isfinite(total):
            raise FloatingPointError('Nonfinite FB loss')
        self.forward_opt.zero_grad(set_to_none=True)
        self.backward_opt.zero_grad(set_to_none=True)
        total.backward()
        f_grad = self.grad_norm(self.f_params)
        b_grad = self.grad_norm(self.net.b.parameters())
        if not torch.isfinite(f_grad+b_grad):
            raise FloatingPointError('Nonfinite F/B gradients')
        self.forward_opt.step(); self.backward_opt.step()
        self.forward_opt.zero_grad(set_to_none=True)
        self.backward_opt.zero_grad(set_to_none=True)
        self.actor_opt.zero_grad(set_to_none=True)
        with frozen_forward(self.net):
            action = self._action(self.net.actor(x,z))
            # B02: keep a direct logit path so saturation does not erase the actor gradient.
            logit = self.net.actor.logits(x,z)
            hinge = torch.relu(logit.abs() - 3.0).mean()
            af = torch.stack((self.net.f1(y,action,z),self.net.f2(y,action,z)))
            policy = actor_loss(af,z) + 1e-3 * hinge
            if not torch.isfinite(policy):
                raise FloatingPointError('Nonfinite Actor loss')
            policy.backward()
            actor_grad = self.grad_norm(self.net.actor.parameters())
            if not torch.isfinite(actor_grad):
                raise FloatingPointError('Nonfinite Actor gradients')
            self.actor_opt.step()
        self.polyak(); self.updates += 1
        return {k:float(v) for k,v in dict(fb=total.detach(),fb_offdiag=offdiag.detach(),
            fb_positive=positive.detach(),orth=orth.detach(),actor=policy.detach(),f_grad=f_grad,
            b_grad=b_grad,actor_grad=actor_grad,f_rms=forwards.detach().square().mean().sqrt(),
            b_norm=backward.detach().norm(dim=-1).mean(),target_density_rms=target.square().mean().sqrt(),
            goal_fraction=self.last_goal_fraction,updates=self.updates).items()}

    def state_dict(self):
        return copy.deepcopy(dict(version=VERSION,schema_version=SCHEMA_VERSION,schema_hash=SCHEMA_HASH,
            estimator_version=ESTIMATOR_VERSION,p_dim=self.p_dim,config=asdict(self.config),
            net=self.net.state_dict(),target=self.target.state_dict(),forward_opt=self.forward_opt.state_dict(),
            backward_opt=self.backward_opt.state_dict(),actor_opt=self.actor_opt.state_dict(),
            generator=self.generator.get_state(),updates=self.updates))

    def load_state_dict(self,state):
        required = dict(version=VERSION,schema_version=SCHEMA_VERSION,schema_hash=SCHEMA_HASH,
                        estimator_version=ESTIMATOR_VERSION,p_dim=self.p_dim,config=asdict(self.config))
        for key,value in required.items():
            if state.get(key) != value:
                raise ValueError(f'Refuse incompatible V2/foreign checkpoint: {key}')
        self.net.load_state_dict(state['net'],strict=True)
        self.target.load_state_dict(state['target'],strict=True)
        self.target.requires_grad_(False)
        for name in ('forward_opt','backward_opt','actor_opt'):
            getattr(self,name).load_state_dict(state[name])
        self.generator.set_state(state['generator'].cpu())
        self.updates = int(state['updates'])


In [ ]:
import os
os.chdir('/content/codebase/unit_bfm')
print(os.getcwd())


In [ ]:
%%writefile __init__.py
"""Single motor / two-link FB research implementation."""

VERSION = "SIM_PRIVILEGED_FF_MR_RF_V2"


In [ ]:
import os
os.chdir('/content/stage2')
print(os.getcwd())


In [ ]:
%%writefile __init__.py
"""Stage-2 physics and learning-curve suite for motor-unit-representation.

Hypotheses stay Candidate until full-scale gates pass. This package is a
scaled full-physics smoke runner: every deviation from the charter protocol
is recorded in ``scale_flags``.
"""

from scale_flags import SCALE_FLAGS

__all__ = ["SCALE_FLAGS"]


In [ ]:
%%writefile scale_flags.py
"""Explicit scale flags for the Stage-2 smoke suite.

Charter quantities are listed next to the values this process actually runs.
A numeric PASS is scale-conditional and does not lock H1–H4.
"""

from __future__ import annotations

SCALE_FLAGS = {
    "execution_tier": "scaled_full_physics_smoke",
    "scientific_lock_allowed": False,
    "plant": "generic_two_link_rotary_unit_and_limb2_v1_smoke",
    "charter_geometry_seeds": 36,
    "charter_stroke_profiles": 4,
    "charter_configs": 144,
    "charter_scenes": 5,
    "charter_joint_cells": 720,
    "executed_geometry_seeds": 36,
    "executed_stroke_profiles": 4,
    "executed_configs": 144,
    "executed_scenes": 5,
    "executed_joint_cells": 720,
    "episode_steps": 80,
    "warmup_steps": 30,
    "rule_inner_servo": "physics_substep",
    "logged_control_hz": 50.0,
    "control_dt_s": 0.02,
    "control_hz": 50.0,
    "charter_physics_dt_s": 2.5e-4,
    "coverage_physics_dt_s": 5.0e-3,
    "coverage_substeps_per_control": 4,
    "halving_probe_dt_s": 2.5e-4,
    "halving_probe_half_dt_s": 1.25e-4,
    "halving_probe_duration_s": 0.04,
    "h4_physics_dt_s": 5.0e-3,
    "h4_substeps_per_control": 4,
    "h4_episode_steps": 16,
    "h4_updates": 24,
    "h4_log_every": 2,
    "h4_task": "hold_at_target_under_pelvis_payload",
    "h4_init_offset_rad": [0.08, -0.06],
    "h4_q_command": "absolute_setpoint_held_over_the_control_step",
    "h4_servo": "torque_pd_recomputed_each_physics_substep",
    "h4_learning_damp": 0.15,
    "h4_sphere_calibration": "unused_latent_bias_1.5_so_decoded_trim_starts_near_zero",
    "charter_upper_trainable_params": 140_000,
    "charter_local_deployed_params": 281_617,
    "charter_online_params": 5_891_633,
    "local_unit": "analytic_frozen_inertia_scaled_pd_on_spherical_z",
    "p_dim_charter": 5065,
    "p_dim_executed": 24,
    "x_dim_charter": 137,
    "x_dim_executed": 16,
    "g_dim": 13,
    "z_dim": 16,
    "z_sphere_radius": 4.0,
    "h1_f_form": "small_mlp_on_standardized_features",
    "h1_mlp_width": 128,
    "h1_mlp_steps": 80,
    "h1_target": "kinematic_block_cos_sin_qd_tip_p_tip_vxy",
    "h3_train_steps": 280,
    "h3_batch": 512,
    "h3_width": 96,
    "h3_erank_floor": 8.0,
    "h3_charter_erank_floor": 12.0,
    "h3_tip_rmse_max": 0.15,
    "h3_holdout_geometry_seeds": [2, 9, 15, 21, 28, 34],
    "h4_upper_width": 48,
    "grid_L2_m": [0.18, 0.25, 0.32],
    "grid_C02_nm": [4.5, 6.0, 9.0],
    "grid_C01_nm": 9.0,
    "stress_L2_m": 0.40,
    "stress_C02_nm": 3.0,
    "limb_m_payload_kg": 2.2,
    "limb_q_target": [0.55, -0.9],
    "limb_m1_kg": 1.0,
    "limb_m2_kg": 0.8,
    "limb_l1_m": 0.25,
    "publish_reserve_lambda": 0.92,
    "edge_reserve": 0.75,
    "sustained_load_min_steps": 25,
    "recovery_min_steps": 20,
    "zero_vel_rmse_max": 0.35,
    "h1_retention_min": 0.95,
    "h3_worst_group_rel_err_max": 0.20,
    "inertia_model_h4": "fixed_hip_planar_rr",
    "load_model_h4": "pelvis_force_through_foot_contact_jacobian",
    "h4_learning_contact": "smooth_viscous_damping",
    "spec_plant_contact": "stateful_bristle_plus_joint_stop",
    "notes": (
        "Coverage rollouts use the full 144 x 5 cell array on a real integrator. "
        "Privileged width, proprioceptive history, network width, upper-parameter "
        "budget, and the learning-loop timestep are scaled down. The 0.25 ms "
        "timestep is used for the halving probe, not for the learning loop."
    ),
}


In [ ]:
%%writefile physics.py
"""Generic single-actuator plant and LIMB2-v1 two-joint smoke plant.

Angle convention for the single rotary unit: housing frame M, rotation about
z_M, q = 0 along +x_M. Gravity in M is (0, -g, 0).

Angle convention for the two-joint limb: q = 0 is upright, positive q1 swings
the hip in the sagittal plane. Link gravity uses analytic dV/dq. A pelvis
payload is a vertical force at the hip, mapped through the foot-contact
Jacobian. It is not a mass on the grounded foot.
"""

from __future__ import annotations

import math
from typing import Dict, Tuple

import torch

G = 9.81


def five_point_accel_coeffs(dt: float, device=None, dtype=None) -> torch.Tensor:
    """Causal quadratic fit over five control samples; returns qdd = 2 c2."""
    t = torch.arange(5, device=device, dtype=torch.float64) * float(dt)
    design = torch.stack([torch.ones(5, dtype=torch.float64), t, t * t], dim=1)
    coef = 2.0 * torch.linalg.pinv(design)[2]
    return coef.to(device=device, dtype=dtype or torch.float32)


def project_to_sphere(x: torch.Tensor, radius: float = 4.0, eps: float = 1e-8) -> torch.Tensor:
    norm = torch.linalg.vector_norm(x, ord=2, dim=-1, keepdim=True).clamp_min(eps)
    return radius * x / norm


def single_link_gravity(q: torch.Tensor, m: torch.Tensor, length: torch.Tensor) -> torch.Tensor:
    """Generalized gravity torque, consistent with V = -m g lc sin(q), lc = L/2.

    With gravity along -y_M and the tip at (L cos q, L sin q, 0):
    V = m * (0, -g, 0) · (lc cos q, lc sin q, 0) = -m g lc sin(q),
    G = dV/dq = -m g lc cos(q).
    Plant uses M qdd = u - G + tau_load, so the cancelling feedforward is +G? 
    Wait: standard manipulator form is M qdd + G = u + tau_load with G = dV/dq.
    Then u_ff = G - tau_load. G = -m g lc cos(q).
    """
    lc = 0.5 * length
    return -m * G * lc * torch.cos(q)


def single_joint_substep(
    q: torch.Tensor,
    qd: torch.Tensor,
    u: torch.Tensor,
    tau_load: torch.Tensor,
    bristle: torch.Tensor,
    m: torch.Tensor,
    length: torch.Tensor,
    inertia: torch.Tensor,
    qmin: torch.Tensor,
    qmax: torch.Tensor,
    dt: float,
    k_stop: float = 40.0,
    d_stop: float = 2.0,
    k_bristle: float = 0.35,
    bristle_max: float = 0.04,
    viscous: float = 0.12,
) -> Tuple[torch.Tensor, ...]:
    """One semi-implicit Euler substep with joint-stop spring, viscous drag, and bristle friction."""
    grav = single_link_gravity(q, m, length)
    pen_hi = torch.relu(q - qmax)
    pen_lo = torch.relu(qmin - q)
    active = (pen_hi + pen_lo) > 0
    tau_stop = -k_stop * pen_hi + k_stop * pen_lo - d_stop * qd * active.to(q.dtype)
    bristle = torch.clamp(bristle + qd * dt, -bristle_max, bristle_max)
    tau_fric = -k_bristle * bristle - viscous * qd
    # M qdd + G = u + tau_load + tau_stop + tau_fric, G = grav
    qdd = (u - grav + tau_load + tau_stop + tau_fric) / inertia.clamp_min(1e-6)
    qd = torch.clamp(qd + dt * qdd, -8.0, 8.0)
    q = q + dt * qd
    return q, qd, bristle, qdd, grav, tau_fric, tau_stop


def tip_kinematics(
    q: torch.Tensor,
    qd: torch.Tensor,
    qdd: torch.Tensor,
    length: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Tip pose/velocity/acceleration in housing frame M. z components are structural zeros."""
    c = torch.cos(q)
    s = torch.sin(q)
    px = length * c
    py = length * s
    pz = torch.zeros_like(px)
    vx = -length * s * qd
    vy = length * c * qd
    vz = torch.zeros_like(vx)
    ax = -length * c * qd * qd - length * s * qdd
    ay = -length * s * qd * qd + length * c * qdd
    az = torch.zeros_like(ax)
    pos = torch.stack([px, py, pz], dim=-1)
    vel = torch.stack([vx, vy, vz], dim=-1)
    acc = torch.stack([ax, ay, az], dim=-1)
    return pos, vel, acc


def pack_g13(
    q: torch.Tensor,
    qd: torch.Tensor,
    qdd_hat: torch.Tensor,
    length: torch.Tensor,
    u_nm: torch.Tensor,
    s_max: torch.Tensor,
) -> torch.Tensor:
    """13-field outcome: cos, sin, qd, qdd_hat, tip p/v/a, with tip z structural 0.

    Torque effort is not a 14th field. The schema note allows a variant to swap
    one tip channel; this smoke keeps the kinematic 13 and reports u_clip beside it.
    """
    pos, vel, acc = tip_kinematics(q, qd, qdd_hat, length)
    g = torch.cat(
        [
            torch.cos(q).unsqueeze(-1),
            torch.sin(q).unsqueeze(-1),
            qd.unsqueeze(-1),
            qdd_hat.unsqueeze(-1),
            pos,
            vel,
            acc,
        ],
        dim=-1,
    )
    assert g.shape[-1] == 13
    return g


def u_clip(u_nm: torch.Tensor, s_max: torch.Tensor) -> torch.Tensor:
    return torch.clamp(u_nm / s_max.clamp_min(1e-6), -1.0, 1.0)


# ---------------------------------------------------------------------------
# LIMB2-v1 two-joint dynamics
# ---------------------------------------------------------------------------


def _as_batch(value, ref: torch.Tensor) -> torch.Tensor:
    if not torch.is_tensor(value):
        value = torch.tensor(value, device=ref.device, dtype=ref.dtype)
    else:
        value = value.to(device=ref.device, dtype=ref.dtype)
    if value.ndim == 0:
        value = value.expand(ref.shape[0])
    return value


def limb_inertial_params(m1, m2, l1, l2):
    lc1 = l1 * 0.5
    lc2 = l2 * 0.5
    i1 = m1 * l1 * l1 / 12.0
    i2 = m2 * l2 * l2 / 12.0
    return lc1, lc2, i1, i2


def limb_mass_matrix(q2, m1, m2, l1, l2):
    """Planar RR inertia. Returns M11, M12, M22, a2 (coupling coefficient)."""
    lc1, lc2, i1, i2 = limb_inertial_params(m1, m2, l1, l2)
    a1 = i1 + i2 + m1 * lc1 * lc1 + m2 * (l1 * l1 + lc2 * lc2)
    a2 = m2 * l1 * lc2
    a3 = i2 + m2 * lc2 * lc2
    c2 = torch.cos(q2)
    m11 = a1 + 2.0 * a2 * c2
    m12 = a3 + a2 * c2
    if not torch.is_tensor(a3):
        m22 = torch.ones_like(m11) * float(a3)
    elif a3.ndim == 0:
        m22 = torch.ones_like(m11) * a3
    else:
        m22 = a3
    return m11, m12, m22, a2


def limb_coriolis(q2, qd1, qd2, a2):
    h = -a2 * torch.sin(q2)
    c1 = h * (2.0 * qd1 * qd2 + qd2 * qd2)
    c2 = -h * qd1 * qd1
    return c1, c2


def limb_gravity(q1, q2, m1, m2, l1, l2):
    """G = dV/dq with z up and q = 0 upright."""
    lc1 = 0.5 * l1
    lc2 = 0.5 * l2
    s1 = torch.sin(q1)
    s12 = torch.sin(q1 + q2)
    g1 = -G * ((m1 * lc1 + m2 * l1) * s1 + m2 * lc2 * s12)
    g2 = -G * (m2 * lc2 * s12)
    return g1, g2


def limb_potential(q1, q2, m1, m2, l1, l2):
    lc1 = 0.5 * l1
    lc2 = 0.5 * l2
    z1 = lc1 * torch.cos(q1)
    z2 = l1 * torch.cos(q1) + lc2 * torch.cos(q1 + q2)
    return G * (m1 * z1 + m2 * z2)


def pelvis_load_torque(q1, q2, l1, l2, m_payload: float):
    """Generalized torque of a vertical pelvis force through the foot Jacobian.

    Foot is the contact origin. Hip (pelvis) position is the forward map of the
    two links. F = (0, -m_payload * g) is applied at the pelvis, never as an
    added foot mass.
    """
    s1 = torch.sin(q1)
    c1 = torch.cos(q1)
    s12 = torch.sin(q1 + q2)
    c12 = torch.cos(q1 + q2)
    dx1 = l1 * c1 + l2 * c12
    dx2 = l2 * c12
    dz1 = -l1 * s1 - l2 * s12
    dz2 = -l2 * s12
    fz = -float(m_payload) * G
    # Fx = 0
    return dz1 * fz, dz2 * fz


def limb_qdd(q1, q2, qd1, qd2, u1, u2, m1, m2, l1, l2, m_payload: float, damp: float = 0.08):
    m11, m12, m22, a2 = limb_mass_matrix(q2, m1, m2, l1, l2)
    c1, c2 = limb_coriolis(q2, qd1, qd2, a2)
    g1, g2 = limb_gravity(q1, q2, m1, m2, l1, l2)
    p1, p2 = pelvis_load_torque(q1, q2, l1, l2, m_payload)
    rhs1 = u1 + p1 - c1 - g1 - damp * qd1
    rhs2 = u2 + p2 - c2 - g2 - damp * qd2
    det = (m11 * m22 - m12 * m12).clamp_min(1e-8)
    qdd1 = (m22 * rhs1 - m12 * rhs2) / det
    qdd2 = (-m12 * rhs1 + m11 * rhs2) / det
    return qdd1, qdd2


def limb_substep(q, qd, u, m1, m2, l1, l2, m_payload: float, dt: float, damp: float = 0.08):
    """Semi-implicit Euler. q, qd, u are [B, 2]. l2 may be [B] or scalar."""
    q1, q2 = q[:, 0], q[:, 1]
    qd1, qd2 = qd[:, 0], qd[:, 1]
    u1, u2 = u[:, 0], u[:, 1]
    l2b = _as_batch(l2, q1)
    qdd1, qdd2 = limb_qdd(q1, q2, qd1, qd2, u1, u2, m1, m2, l1, l2b, m_payload, damp=damp)
    qd_next = torch.stack(
        [
            torch.clamp(qd1 + dt * qdd1, -8.0, 8.0),
            torch.clamp(qd2 + dt * qdd2, -8.0, 8.0),
        ],
        dim=-1,
    )
    q_next = q + dt * qd_next
    return q_next, qd_next


def static_required_torque(q1, q2, m1, m2, l1, l2, m_payload: float):
    """Inverse-dynamics torque at rest: u = G - tau_pelvis (C = 0, no damping)."""
    g1, g2 = limb_gravity(q1, q2, m1, m2, l1, l2)
    p1, p2 = pelvis_load_torque(q1, q2, l1, l2, m_payload)
    # At rest rhs without u is p - g, and M qdd = u + p - g, so u = g - p
    return g1 - p1, g2 - p2


def forward_kinematics(q1, q2, l1, l2):
    x = l1 * torch.sin(q1) + l2 * torch.sin(q1 + q2)
    z = l1 * torch.cos(q1) + l2 * torch.cos(q1 + q2)
    return x, z


def inverse_kinematics(x, z, l1, l2):
    """Elbow-bent (q2 <= 0) analytic IK. Targets are clamped into the reachable annulus."""
    l2b = _as_batch(l2, x) if not torch.is_tensor(l2) or l2.ndim == 0 else l2
    if not torch.is_tensor(l1):
        l1t = torch.ones_like(x) * float(l1)
    else:
        l1t = l1.to(dtype=x.dtype, device=x.device)
        if l1t.ndim == 0:
            l1t = l1t.expand_as(x)
    r_max = (l1t + l2b) - 1e-6
    r_min = (l1t - l2b).abs() + 1e-6
    r = torch.sqrt(x * x + z * z).clamp(min=1e-8)
    r_clamped = torch.minimum(torch.maximum(r, r_min), r_max)
    # keep direction
    x = x / r * r_clamped
    z = z / r * r_clamped
    r2 = x * x + z * z
    c2 = (r2 - l1t * l1t - l2b * l2b) / (2.0 * l1t * l2b)
    c2 = c2.clamp(-1.0 + 1e-6, 1.0 - 1e-6)
    q2 = -torch.acos(c2)
    alpha = torch.atan2(l2b * torch.sin(q2), l1t + l2b * torch.cos(q2))
    q1 = torch.atan2(x, z) - alpha
    return q1, q2


def schur_identity_error(q2: float = 0.4, m1: float = 1.0, m2: float = 0.8, l1: float = 0.25, l2: float = 0.25) -> float:
    """Compare full inertia solve at tau=(1, 0) with the Schur complement of joint 2."""
    q2t = torch.tensor([q2], dtype=torch.float64)
    m11, m12, m22, _ = limb_mass_matrix(q2t, m1, m2, l1, l2)
    det = m11 * m22 - m12 * m12
    # inv(M) @ [1, 0]
    qdd1 = m22 / det
    m_schur = m11 - (m12 * m12) / m22
    qdd1_s = 1.0 / m_schur
    return float((qdd1 - qdd1_s).abs().item())


def gravity_identity_error(q1: float = 0.4, q2: float = -0.7) -> float:
    """Finite-difference dV/dq versus analytic gravity."""
    m1, m2, l1, l2 = 1.0, 0.8, 0.25, 0.22
    eps = 1e-6
    q1t = torch.tensor(q1, dtype=torch.float64)
    q2t = torch.tensor(q2, dtype=torch.float64)
    g1, g2 = limb_gravity(q1t, q2t, m1, m2, l1, l2)

    def V(a, b):
        return limb_potential(a, b, m1, m2, l1, l2)

    g1_n = (V(q1t + eps, q2t) - V(q1t - eps, q2t)) / (2 * eps)
    g2_n = (V(q1t, q2t + eps) - V(q1t, q2t - eps)) / (2 * eps)
    return float(torch.max(torch.stack([(g1 - g1_n).abs(), (g2 - g2_n).abs()])).item())


def free_flight_com_identity_error(q1: float = 0.3, q2: float = -0.5) -> float:
    """Internal-torque-free check: G matches the COM Jacobian transpose times total weight.

    For this fixed-base model the identity is G(q) = J_com(q)^T (m_tot * g_vec)
    with g_vec the upward coordinate derivative convention used in V.
    """
    m1, m2, l1, l2 = 1.1, 0.7, 0.25, 0.25
    q1t = torch.tensor(q1, dtype=torch.float64, requires_grad=True)
    q2t = torch.tensor(q2, dtype=torch.float64, requires_grad=True)
    # COM height
    lc1 = 0.5 * l1
    lc2 = 0.5 * l2
    z1 = lc1 * torch.cos(q1t)
    z2 = l1 * torch.cos(q1t) + lc2 * torch.cos(q1t + q2t)
    z_com = (m1 * z1 + m2 * z2) / (m1 + m2)
    # d z_com / dq * (m_tot * g) should equal G
    dz1, dz2 = torch.autograd.grad(z_com, [q1t, q2t])
    g1, g2 = limb_gravity(q1t.detach(), q2t.detach(), m1, m2, l1, l2)
    pred1 = dz1 * (m1 + m2) * G
    pred2 = dz2 * (m1 + m2) * G
    return float(torch.max((pred1 - g1).abs(), (pred2 - g2).abs()).item())


def ik_roundtrip_error() -> float:
    q1 = torch.tensor([0.45, -0.2], dtype=torch.float64)
    q2 = torch.tensor([-0.8, -0.4], dtype=torch.float64)
    l1, l2 = 0.25, 0.25
    x, z = forward_kinematics(q1, q2, l1, l2)
    q1b, q2b = inverse_kinematics(x, z, l1, l2)
    return float(torch.max(torch.stack([(q1 - q1b).abs(), (q2 - q2b).abs()])).item())


def halving_gate(dt: float = 2.5e-4, duration: float = 0.04) -> Dict[str, float]:
    """Float64 semi-implicit Euler at dt and dt/2. Relative final-state drift."""
    m1, m2, l1, l2 = 1.0, 0.8, 0.25, 0.25
    m_payload = 1.5

    def roll(step):
        q = torch.tensor([[0.2, -0.5]], dtype=torch.float64)
        qd = torch.tensor([[0.0, 0.0]], dtype=torch.float64)
        n = int(round(duration / step))
        impulse = 0.0
        for i in range(n):
            phase = i * step
            u = torch.tensor(
                [[1.5 * math.sin(8 * phase), -0.8 * math.cos(5 * phase)]],
                dtype=torch.float64,
            )
            q, qd = limb_substep(q, qd, u, m1, m2, l1, l2, m_payload, step, damp=0.05)
            impulse += float(u.abs().sum().item()) * step
        return q, qd, impulse

    q_a, qd_a, imp_a = roll(dt)
    q_b, qd_b, imp_b = roll(dt * 0.5)
    dq = float((q_a - q_b).abs().max().item())
    dqv = float((qd_a - qd_b).abs().max().item())
    # impulses of the open-loop command match by construction; compare state drift
    # relative to the coarser trajectory magnitude
    mag = float(q_a.abs().max().item()) + float(qd_a.abs().max().item()) + 1e-8
    rel = max(dq, dqv) / mag
    return {
        "dt_s": dt,
        "half_dt_s": dt * 0.5,
        "max_abs_q_drift": dq,
        "max_abs_qd_drift": dqv,
        "relative_drift": rel,
        "command_impulse_coarse": imp_a,
        "command_impulse_fine": imp_b,
        "pass": rel < 0.05,
    }


def build_unit_configs(dtype=torch.float64):
    """36 geometry seeds x 4 stroke profiles. All-feasible under the smoke load law."""
    n_seeds = 36
    lengths = torch.linspace(0.12, 0.32, n_seeds, dtype=dtype)
    masses = 0.55 + 0.004 * torch.arange(n_seeds, dtype=dtype)
    c0_cycle = torch.tensor([3.0, 6.0, 9.0], dtype=dtype)
    strokes = [
        {"qmin": -1.6, "qmax": 1.6, "amp": 0.0, "freq": 0.0},
        {"qmin": -1.8, "qmax": 1.5, "amp": 0.25, "freq": 0.4},
        {"qmin": -2.0, "qmax": 1.8, "amp": 0.40, "freq": 0.7},
        {"qmin": -2.2, "qmax": 2.0, "amp": 0.55, "freq": 1.0},
    ]
    rows = []
    for s in range(n_seeds):
        for k, stroke in enumerate(strokes):
            c0 = c0_cycle[s % 3]
            length = lengths[s]
            mass = masses[s]
            j_link = mass * length * length / 3.0
            gear = 8.0
            j_rotor = 3.0e-4
            s_max = float(1.25 * c0)
            # Scale reflected inertia with actuator capacity so a feasible load
            # (a fixed fraction of S_max) produces a comparable angular acceleration.
            inertia = max(float(j_link + (gear ** 2) * j_rotor), 0.045 * s_max)
            rows.append(
                {
                    "config_id": s * 4 + k,
                    "seed_id": s,
                    "stroke_id": k,
                    "length": float(length),
                    "mass": float(mass),
                    "c0": float(c0),
                    "s_max": s_max,
                    "inertia": float(inertia),
                    "gear": gear,
                    "qmin": stroke["qmin"],
                    "qmax": stroke["qmax"],
                    "amp": stroke["amp"],
                    "freq": stroke["freq"],
                }
            )
    assert len(rows) == 144
    return rows


In [ ]:
%%writefile contracts.py
"""Re-export of the B01–B04 merge norms.

The implementation lives in ``unit_bfm.v3.stage2_norms`` so the V3 learner
and this gate suite share one contract. Sibling import style is preserved
for ``python run_stage2.py``.
"""

from __future__ import annotations

import sys
from pathlib import Path

_CODEBASE = Path(__file__).resolve().parents[1] / "codebase"
if str(_CODEBASE) not in sys.path:
    sys.path.insert(0, str(_CODEBASE))

from unit_bfm.v3.stage2_norms import (  # noqa: E402
    FUTURE_SLICE,
    LAGGED_SLICE,
    P_DIM,
    SCENE_ID,
    SCENES,
    VARIANTS,
    DualPartitionNormalizer,
    SupportSampler,
    demand_label,
    equilibrium_label,
    frozen_local_hash,
    logit_path_alive,
    make_program,
    marginals_do_not_certify_joint,
    matched_pair_mask,
    nondim_v3_g,
    participation_ratio,
    philox_digest,
    rule_step,
    sanitize_privileged,
    sanitize_privileged_tensor,
    schema_guard_g,
    single_link_gravity,
    t_kappa,
    t_kappa_inverse,
)

__all__ = [
    "FUTURE_SLICE",
    "LAGGED_SLICE",
    "P_DIM",
    "SCENE_ID",
    "SCENES",
    "VARIANTS",
    "DualPartitionNormalizer",
    "SupportSampler",
    "demand_label",
    "equilibrium_label",
    "frozen_local_hash",
    "logit_path_alive",
    "make_program",
    "marginals_do_not_certify_joint",
    "matched_pair_mask",
    "nondim_v3_g",
    "participation_ratio",
    "philox_digest",
    "rule_step",
    "sanitize_privileged",
    "sanitize_privileged_tensor",
    "schema_guard_g",
    "single_link_gravity",
    "t_kappa",
    "t_kappa_inverse",
]


In [ ]:
%%writefile suite.py
"""Stage-2 gates H1–H4 on the scaled full-physics smoke plant.

Numeric PASS/FAIL values are scale-conditional. ``scientific_lock`` stays false.
"""

from __future__ import annotations

import math
import time
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn

from contracts import (
    FUTURE_SLICE,
    P_DIM,
    DualPartitionNormalizer,
    SupportSampler,
    equilibrium_label,
    frozen_local_hash,
    logit_path_alive,
    make_program,
    marginals_do_not_certify_joint,
    participation_ratio,
    philox_digest,
    rule_step,
    sanitize_privileged,
    schema_guard_g,
    t_kappa,
    t_kappa_inverse,
)
from physics import (
    build_unit_configs,
    five_point_accel_coeffs,
    forward_kinematics,
    free_flight_com_identity_error,
    gravity_identity_error,
    halving_gate,
    ik_roundtrip_error,
    inverse_kinematics,
    limb_gravity,
    limb_mass_matrix,
    limb_substep,
    pack_g13,
    pelvis_load_torque,
    project_to_sphere,
    schur_identity_error,
    single_joint_substep,
    single_link_gravity,
    static_required_torque,
    u_clip,
)
from scale_flags import SCALE_FLAGS

FROZEN_GAINS = {
    "kp": 27.0,
    "ki": 27.0,
    "kd": 9.0,
    "lambda_reserve": 0.92,
    "z_radius": 4.0,
    # Limb servo gains are torques, not accelerations. Distal inertia is too
    # small for an inertia-scaled acceleration gain to both beat the pose
    # stiffness and stay inside a 5 ms step.
    "h4_kp": 20.0,
    "h4_kd": 1.2,
    "h4_q_scale": 0.35,
    "h4_ff_scale": 0.35,
}


def _device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _py(v):
    if isinstance(v, dict):
        return {str(k): _py(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)):
        return [_py(x) for x in v]
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if isinstance(v, torch.Tensor):
        if v.ndim == 0:
            return _py(v.detach().cpu().item())
        return _py(v.detach().cpu().tolist())
    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return None
        return v
    return v


def spec_closure() -> Dict:
    """Controller-independent identities and contract unit checks."""
    schur = schur_identity_error()
    grav = gravity_identity_error()
    com = free_flight_com_identity_error()
    ik = ik_roundtrip_error()
    half = halving_gate(
        dt=SCALE_FLAGS["halving_probe_dt_s"],
        duration=SCALE_FLAGS["halving_probe_duration_s"],
    )
    d_r = philox_digest(7, 3, "E4", 0, 0, "E4-R")
    d_ri = philox_digest(7, 3, "E4", 0, 0, "E4-RI")
    labels = {
        "over_reserve": equilibrium_label(0.2, 1.0, 0.95, 8.0),
        "under_load": equilibrium_label(0.0, 1.0, 0.10, 8.0),
        "ok": equilibrium_label(0.4, -1.0, 0.55, 8.0),
    }
    # Rule equilibrium: κ=1 and a warm delay cancel a feasible load exactly.
    q_star = torch.tensor([0.70])
    mass = torch.tensor([0.56])
    length = torch.tensor([0.22])
    s_max = torch.tensor([7.5])
    inertia = torch.tensor([0.02])
    grav_star = single_link_gravity(q_star, mass, length)
    sigma = torch.tensor([1.0])
    u_star = sigma * 0.70 * s_max
    tau_load = grav_star - u_star
    u, _ = rule_step(
        q_star,
        torch.zeros(1),
        q_star,
        torch.zeros(1),
        inertia,
        grav_star,
        tau_load,
        torch.ones(1),
        s_max,
        0.02,
        kp=FROZEN_GAINS["kp"],
        ki=FROZEN_GAINS["ki"],
    )
    residual = float((u - grav_star + tau_load).abs().item())
    # Sanitizer strips a future-dated slice.
    raw = torch.zeros(4, P_DIM)
    raw[:, 12:16] = 0.2
    raw[:, 16:20] = 3.0  # endogenous future, not equal to the lag
    cleaned = sanitize_privileged(raw)
    g = torch.zeros(5, 13)
    g[:, 0] = 1.0
    g[:, 4] = 0.2
    g[:, 7] = 0.1
    rho = torch.full((5,), 0.2)
    g_hat = t_kappa(g, rho)
    g_back = t_kappa_inverse(g_hat)
    # vz was structurally 0 and is the slot λ overwrites
    roundtrip = float((g_back - g).abs().max().item())
    sampler = SupportSampler({0: 10, 1: 0, 2: 8})
    missing = sampler.sample([0, 1])
    ok_w = sampler.sample([0, 2])
    hinge = logit_path_alive()
    norm = DualPartitionNormalizer(4, 4)
    norm.update_proprio(torch.ones(8, 4))
    snap = norm.p_mean.clone()
    norm.freeze()
    norm.update_proprio(torch.ones(8, 4) * 9)
    norm.update_privileged(torch.ones(8, 4) * 9)
    freeze_ok = float((norm.p_mean - snap).abs().max().item()) == 0.0 and norm.privileged_updates == 0
    # Partition isolation while unfrozen: privileged updates must not move proprio stats.
    norm2 = DualPartitionNormalizer(4, 4)
    norm2.update_proprio(torch.randn(16, 4))
    snap2 = norm2.p_mean.clone()
    norm2.update_privileged(torch.randn(16, 4) * 5)
    isolated = float((norm2.p_mean - snap2).abs().max().item()) == 0.0
    passed = all(
        [
            schur < 1e-8,
            grav < 1e-5,
            com < 1e-5,
            ik < 1e-4,
            bool(half["pass"]),
            d_r != d_ri,
            labels["over_reserve"] == "INFEASIBLE_CONFIG",
            labels["under_load"] == "INFEASIBLE_CONFIG",
            labels["ok"] == "FEASIBLE",
            residual < 1e-5,
            cleaned["consumed_violations"] == 0,
            cleaned["raw_fail_closed_rows"] == 4,
            cleaned["freeze_normalizer"] is True,
            roundtrip < 1e-5,
            schema_guard_g(g) == "OK",
            missing["status"] == "INSUFFICIENT_SUPPORT",
            ok_w["status"] == "OK" and abs(float(np.sum(ok_w["weights"])) - 1.0) < 1e-8,
            hinge["pass"],
            marginals_do_not_certify_joint(),
            freeze_ok,
            isolated,
        ]
    )
    return {
        "pass": passed,
        "schur_identity_abs_err": schur,
        "gravity_finite_difference_abs_err": grav,
        "free_flight_com_identity_abs_err": com,
        "ik_roundtrip_abs_err": ik,
        "halving_gate": half,
        "e4_r_digest": d_r,
        "e4_ri_digest": d_ri,
        "e4_keys_distinct": d_r != d_ri,
        "equilibrium_labels": labels,
        "rule_equilibrium_residual_nm": residual,
        "t_kappa_roundtrip_abs_err": roundtrip,
        "logit_path": hinge,
        "marginals_insufficient": marginals_do_not_certify_joint(),
        "fail_closed_freeze": freeze_ok,
        "bn_partitions_isolated": isolated,
        "frozen_hash": frozen_local_hash(FROZEN_GAINS),
        "frozen_hash_check": frozen_local_hash(FROZEN_GAINS),
    }


def _stack_configs(configs: List[dict], device, dtype):
    def col(key):
        return torch.tensor([c[key] for c in configs], device=device, dtype=dtype)

    return {
        "length": col("length"),
        "mass": col("mass"),
        "s_max": col("s_max"),
        "inertia": col("inertia"),
        "qmin": col("qmin"),
        "qmax": col("qmax"),
        "amp": col("amp"),
        "freq": col("freq"),
        "c0": col("c0"),
        "seed_id": col("seed_id").long(),
        "stroke_id": col("stroke_id").long(),
        "config_id": col("config_id").long(),
    }


def rollout_coverage(configs: List[dict]) -> Dict[str, np.ndarray]:
    """One physics episode per (config, scene) cell. Returns flat step records."""
    device = torch.device("cpu")
    dtype = torch.float32
    steps = int(SCALE_FLAGS["episode_steps"])
    warmup = int(SCALE_FLAGS["warmup_steps"])
    dt_c = float(SCALE_FLAGS["control_dt_s"])
    n_sub = int(SCALE_FLAGS["coverage_substeps_per_control"])
    dt = float(SCALE_FLAGS["coverage_physics_dt_s"])
    n_cfg = len(configs)
    n_scene = 5
    B = n_cfg * n_scene
    cfg = _stack_configs(configs, device, dtype)
    # Repeat each config across scenes. scene index cycles fastest.
    rep = n_scene
    length = cfg["length"].repeat_interleave(rep)
    mass = cfg["mass"].repeat_interleave(rep)
    s_max = cfg["s_max"].repeat_interleave(rep)
    inertia = cfg["inertia"].repeat_interleave(rep)
    qmin = cfg["qmin"].repeat_interleave(rep)
    qmax = cfg["qmax"].repeat_interleave(rep)
    amp = cfg["amp"].repeat_interleave(rep)
    freq = cfg["freq"].repeat_interleave(rep)
    c0 = cfg["c0"].repeat_interleave(rep)
    seed_id = cfg["seed_id"].repeat_interleave(rep)
    config_id = cfg["config_id"].repeat_interleave(rep)
    scene_id = torch.arange(n_scene, device=device).repeat(n_cfg)

    programs = []
    infeasible = 0
    for c in configs:
        for scene in ("E0", "E1", "E2", "E3", "E4"):
            prog = make_program(
                20261024,
                c["config_id"],
                scene,
                n=0,
                domain=0,
                variant="train",
                length=c["length"],
                mass=c["mass"],
                s_max=c["s_max"],
                qmin=c["qmin"],
                qmax=c["qmax"],
                amp=c["amp"],
                freq=c["freq"],
                steps=steps,
                control_dt=dt_c,
            )
            if prog["status"] != "FEASIBLE":
                infeasible += 1
            programs.append(prog)
    assert len(programs) == B
    tau_cmd = torch.tensor(np.stack([p["tau_load"] for p in programs], axis=0), device=device, dtype=dtype)
    q_des = torch.tensor(np.stack([p["q_des"] for p in programs], axis=0), device=device, dtype=dtype)
    kappa = torch.tensor([p["kappa"] for p in programs], device=device, dtype=dtype)
    q_star = torch.tensor([p["q_star"] for p in programs], device=device, dtype=dtype)
    # Contact site for E2, a fixed exogenous angle.
    q_obs = q_star + 0.12
    k_contact = 5.0 + (config_id % 4).to(dtype)

    q = q_star.clone()
    # E1 starts just inside the positive stop.
    e1 = scene_id == 1
    q = torch.where(e1, qmax - 0.03, q)
    qd = torch.zeros(B, device=device, dtype=dtype)
    integ = torch.zeros(B, device=device, dtype=dtype)
    bristle = torch.zeros(B, device=device, dtype=dtype)
    tau_delay = tau_cmd[:, 0].clone()
    coef = five_point_accel_coeffs(dt_c, device=device, dtype=dtype)
    q_buf = q.unsqueeze(1).repeat(1, 5)

    total = warmup + steps
    store_q = []
    store_qd = []
    store_u = []
    store_tau = []
    store_fric = []
    store_stop = []
    store_qdd = []
    store_grav = []
    finite = torch.ones(B, dtype=torch.bool)

    for t in range(total):
        tau_g_hat = single_link_gravity(q, mass, length)
        src = min(t, steps - 1) if t >= warmup else 0
        # During warmup, hold the first-step command. After warmup, index t-warmup.
        if t < warmup:
            qdes_t = q_des[:, 0]
            cmd_t = tau_cmd[:, 0]
            # E1 warmup should not start on the stop spring already slammed.
        else:
            k = t - warmup
            qdes_t = q_des[:, k]
            cmd_t = tau_cmd[:, k]
        contact = torch.zeros(B, device=device, dtype=dtype)
        if True:
            depth = torch.relu(q - q_obs)
            contact = torch.where(scene_id == 2, -k_contact * depth - 0.4 * qd * (depth > 0), contact)
        tau_plant = cmd_t + contact
        # Inner servo runs at the physics substep. The logged command is the last
        # substep of the 50 Hz bucket. Delayed feedforward still updates once
        # per control step, so E3 keeps a one-step lag and E4 stays κ=0.
        u = torch.zeros(B, device=device, dtype=dtype)
        qdd_last = torch.zeros(B, device=device, dtype=dtype)
        fric_last = torch.zeros(B, device=device, dtype=dtype)
        stop_last = torch.zeros(B, device=device, dtype=dtype)
        grav_last = tau_g_hat
        tau_s = cmd_t
        for _ in range(n_sub):
            tau_g_hat = single_link_gravity(q, mass, length)
            depth = torch.relu(q - q_obs)
            contact_s = torch.where(
                scene_id == 2,
                -k_contact * depth - 0.4 * qd * (depth > 0),
                torch.zeros_like(q),
            )
            tau_s = cmd_t + contact_s
            u, integ = rule_step(
                q, qd, qdes_t, integ, inertia, tau_g_hat, tau_delay, kappa, s_max, dt,
                kp=FROZEN_GAINS["kp"], ki=FROZEN_GAINS["ki"],
            )
            q, qd, bristle, qdd_last, grav_last, fric_last, stop_last = single_joint_substep(
                q, qd, u, tau_s, bristle, mass, length, inertia, qmin, qmax, dt
            )
        tau_delay = cmd_t + torch.where(
            scene_id == 2,
            -k_contact * torch.relu(q - q_obs),
            torch.zeros_like(q),
        )
        q_buf = torch.cat([q_buf[:, 1:], q.unsqueeze(1)], dim=1)
        finite = finite & torch.isfinite(q) & torch.isfinite(qd)
        store_q.append(q)
        store_qd.append(qd)
        store_u.append(u)
        store_tau.append(tau_s.detach())
        store_fric.append(fric_last.detach())
        store_stop.append(stop_last.detach())
        store_qdd.append(qdd_last.detach())
        store_grav.append(grav_last.detach())

    Q = torch.stack(store_q, 0)  # [Ttot, B]
    QD = torch.stack(store_qd, 0)
    U = torch.stack(store_u, 0)
    TAU = torch.stack(store_tau, 0)
    FR = torch.stack(store_fric, 0)
    ST = torch.stack(store_stop, 0)
    QDD = torch.stack(store_qdd, 0)
    # Control-rate q history for the 5-point fit, including warmup.
    # Rebuild q buffer causally from Q.
    qdd_hat = torch.zeros_like(Q)
    for t in range(total):
        window = []
        for lag in range(4, -1, -1):
            idx = max(0, t - lag)
            window.append(Q[idx])
        stacked = torch.stack(window, 0)  # [5, B]
        qdd_hat[t] = torch.einsum("s,sb->b", coef, stacked)

    # Keep post-warmup steps only, but future slots need t+1, so the last step is dropped later.
    sl = slice(warmup, warmup + steps)
    out = {
        "q": Q[sl].numpy(),
        "qd": QD[sl].numpy(),
        "u": U[sl].numpy(),
        "tau": TAU[sl].numpy(),
        "fric": FR[sl].numpy(),
        "stop": ST[sl].numpy(),
        "qdd": QDD[sl].numpy(),
        "qdd_hat": qdd_hat[sl].numpy(),
        "length": length.numpy(),
        "mass": mass.numpy(),
        "s_max": s_max.numpy(),
        "inertia": inertia.numpy(),
        "c0": c0.numpy(),
        "amp": amp.numpy(),
        "freq": freq.numpy(),
        "seed_id": seed_id.numpy(),
        "config_id": config_id.numpy(),
        "scene_id": scene_id.numpy(),
        "kappa": kappa.numpy(),
        "q_des": q_des.numpy(),
        "tau_cmd": tau_cmd.numpy(),
        "qmin": qmin.numpy(),
        "qmax": qmax.numpy(),
        "finite": finite.numpy(),
        "infeasible_programs": infeasible,
        "e3_onset": np.array([p["onset"] for p in programs], dtype=np.int64),
        "e3_len": np.array([p["e3_len"] for p in programs], dtype=np.int64),
        "e4_release": np.array([p["e4_release"] for p in programs], dtype=np.int64),
        "anchor": np.array([p["anchor"] for p in programs], dtype=np.int64),
        "tier": np.array([p["tier"] for p in programs], dtype=np.int64),
        "program_status": np.array([1 if p["status"] == "FEASIBLE" else 0 for p in programs], dtype=np.int64),
    }
    return out


def static_hold_panel(configs: List[dict]) -> Dict[str, float]:
    """Canonical zero-velocity hold, both load signs, plus a u=0 negative control."""
    device = torch.device("cpu")
    dtype = torch.float32
    dt_c = float(SCALE_FLAGS["control_dt_s"])
    n_sub = int(SCALE_FLAGS["coverage_substeps_per_control"])
    dt = float(SCALE_FLAGS["coverage_physics_dt_s"])
    steps = 60
    cfg = _stack_configs(configs, device, dtype)
    # two signs
    signs = (1.0, -1.0)
    masses = []
    rmses = []
    pos_err = []
    control_fail = 0
    for sigma in signs:
        B = len(configs)
        length, mass, s_max = cfg["length"], cfg["mass"], cfg["s_max"]
        inertia, qmin, qmax = cfg["inertia"], cfg["qmin"], cfg["qmax"]
        q_star = torch.full((B,), 0.0, device=device, dtype=dtype)
        grav = single_link_gravity(q_star, mass, length)
        m_frac = (0.25 + grav.abs() / s_max + 0.06).clamp(0.40, 0.70)
        u_star = sigma * m_frac * s_max
        tau_level = grav - u_star
        q = q_star.clone()
        qd = torch.zeros(B, device=device, dtype=dtype)
        integ = torch.zeros(B, device=device, dtype=dtype)
        bristle = torch.zeros(B, device=device, dtype=dtype)
        kappa = torch.ones(B, device=device, dtype=dtype)
        tau_delay = tau_level.clone()
        qd_tail = []
        last_q = q
        finite = True
        for t in range(steps):
            tau_g_hat = single_link_gravity(q, mass, length)
            u, integ = rule_step(
                q, qd, q_star, integ, inertia, tau_g_hat, tau_delay, kappa, s_max, dt_c,
                kp=FROZEN_GAINS["kp"], ki=FROZEN_GAINS["ki"],
            )
            for _ in range(n_sub):
                q, qd, bristle, _, _, _, _ = single_joint_substep(
                    q, qd, u, tau_level, bristle, mass, length, inertia, qmin, qmax, dt
                )
            tau_delay = tau_level
            if t >= steps - 20:
                qd_tail.append(qd.detach().clone())
            last_q = q
            finite = finite and bool(torch.isfinite(q).all() and torch.isfinite(qd).all())
        tail = torch.stack(qd_tail, 0)
        rmse = torch.sqrt(torch.mean(tail * tail, dim=0))
        perr = (last_q - q_star).abs()
        rmses.append(rmse)
        pos_err.append(perr)
        control_fail += int((~torch.isfinite(last_q) | (perr > 0.6)).sum().item())
        if not finite:
            control_fail += B
    rmse_all = torch.cat(rmses, 0)
    perr_all = torch.cat(pos_err, 0)

    # Negative control: u = 0 on the positive-sign load, 16 configs.
    n_neg = 16
    length = cfg["length"][:n_neg]
    mass = cfg["mass"][:n_neg]
    s_max = cfg["s_max"][:n_neg]
    inertia = cfg["inertia"][:n_neg]
    qmin = cfg["qmin"][:n_neg]
    qmax = cfg["qmax"][:n_neg]
    q_star = torch.full((n_neg,), 0.0, dtype=dtype)
    grav = single_link_gravity(q_star, mass, length)
    m_frac = (0.25 + grav.abs() / s_max + 0.06).clamp(0.40, 0.70)
    tau_level = grav - m_frac * s_max
    q = q_star.clone()
    qd = torch.zeros(n_neg, dtype=dtype)
    bristle = torch.zeros(n_neg, dtype=dtype)
    u = torch.zeros(n_neg, dtype=dtype)
    qd_tail = []
    for t in range(40):
        for _ in range(n_sub):
            q, qd, bristle, _, _, _, _ = single_joint_substep(
                q, qd, u, tau_level, bristle, mass, length, inertia, qmin, qmax, dt
            )
        if t >= 20:
            qd_tail.append(qd.detach().clone())
    neg = torch.stack(qd_tail, 0)
    neg_rmse = float(torch.sqrt(torch.mean(neg * neg)).item())
    return {
        "zero_vel_rmse_mean": float(rmse_all.mean().item()),
        "zero_vel_rmse_worst": float(rmse_all.max().item()),
        "hold_position_abs_err_worst": float(perr_all.max().item()),
        "control_fail_count": int(control_fail),
        "negative_control_rmse": neg_rmse,
        "negative_control_position_abs_err": float((q - q_star).abs().mean().item()),
        "n_panel": int(rmse_all.numel()),
    }


def _build_step_tensors(roll: Dict[str, np.ndarray]):
    """Pack transitions (state_t, action that produces state_{t+1}, g_{t+1}).

    Stored rollout rows are post-step. The action aligned with the successor of
    row t is ``u[t+1]``, and the endogenous future slice is the reaction at t+1.
    """
    T, B = roll["q"].shape
    q = roll["q"]
    qd = roll["qd"]
    u = roll["u"]
    length = roll["length"]
    s_max = roll["s_max"]
    qt = torch.tensor(q)
    qdt = torch.tensor(qd)
    qddt = torch.tensor(roll["qdd_hat"])
    lt = torch.tensor(length)
    ut = torch.tensor(u)
    st = torch.tensor(s_max)
    g_all = [pack_g13(qt[t], qdt[t], qddt[t], lt, ut[t], st) for t in range(T)]
    G = torch.stack(g_all, 0).numpy()

    n = T - 1

    def lag_q(lag: int) -> np.ndarray:
        idx = np.clip(np.arange(n) - lag, 0, T - 1)
        return q[idx]

    ones = np.ones((n, B), dtype=np.float64)
    x = np.stack(
        [
            np.cos(q[:n]),
            np.sin(q[:n]),
            q[:n],
            qd[:n],
            lag_q(1),
            lag_q(2),
            lag_q(3),
            lag_q(4),
            u[:n],
            np.broadcast_to(roll["inertia"], (n, B)) * 50.0,
            np.broadcast_to(s_max, (n, B)) / 10.0,
            np.broadcast_to(length, (n, B)),
            np.broadcast_to(roll["mass"], (n, B)),
            np.transpose(roll["q_des"][:, :n], (1, 0)),
            roll["tau"][:n] / np.broadcast_to(s_max, (n, B)),
            ones,
        ],
        axis=-1,
    ).astype(np.float64)

    reaction = roll["tau"] + roll["fric"] + roll["stop"]
    fut = np.stack(
        [reaction[1:], roll["qdd"][1:], roll["fric"][1:], roll["stop"][1:]],
        axis=-1,
    ).astype(np.float64)
    lag = np.zeros_like(fut)
    lag[1:] = fut[:-1]
    lag[0] = fut[0]
    sched = roll["tau_cmd"]
    sched_now = np.transpose(sched[:, :n], (1, 0))
    sched_next = np.transpose(sched[:, 1:], (1, 0))
    p = np.zeros((n, B, P_DIM), dtype=np.float64)
    p[..., 0] = roll["scene_id"] / 4.0
    p[..., 1] = (np.arange(n) / max(n, 1))[:, None]
    p[..., 2] = roll["kappa"]
    p[..., 3] = roll["amp"]
    p[..., 4] = sched_now
    p[..., 5] = sched_next
    p[..., 6] = sched_next
    p[..., 7] = roll["freq"]
    p[..., 8] = roll["tau"][:n]
    p[..., 9] = roll["fric"][:n]
    p[..., 10] = roll["stop"][:n]
    p[..., 11] = roll["qdd_hat"][:n]
    p[..., 12:16] = lag
    p[..., 16:20] = fut
    p[..., 20] = roll["c0"] / 9.0
    p[..., 21] = (roll["scene_id"] == 3).astype(np.float64)

    a = u[1:][..., None].astype(np.float64)
    tau_next = roll["tau"][1:].astype(np.float64)

    def tile(vec):
        return np.broadcast_to(vec, (n, B)).copy()

    flat3 = lambda z: z.reshape(-1, z.shape[-1])
    return {
        "x": flat3(x),
        "p": flat3(p),
        "a": flat3(a),
        "g": flat3(G[:n].astype(np.float64)),
        "g_next": flat3(G[1:].astype(np.float64)),
        "config_id": tile(roll["config_id"]).reshape(-1),
        "seed_id": tile(roll["seed_id"]).reshape(-1),
        "scene_id": tile(roll["scene_id"]).reshape(-1),
        "length": tile(roll["length"]).reshape(-1),
        "mass": tile(roll["mass"]).reshape(-1),
        "c0": tile(roll["c0"]).reshape(-1),
        "amp": tile(roll["amp"]).reshape(-1),
        "freq": tile(roll["freq"]).reshape(-1),
        "s_max": tile(roll["s_max"]).reshape(-1),
        "tau_next": tau_next.reshape(-1),
        "u": a.reshape(-1),
    }


def _explained_variance(pred: np.ndarray, target: np.ndarray) -> float:
    var = target.var(axis=0)
    mask = var > 1e-6
    if not np.any(mask):
        return 0.0
    resid = ((pred - target) ** 2).mean(axis=0)
    ev = 1.0 - resid[mask] / var[mask]
    return float(np.mean(ev))


class _ScaledMLP(nn.Module):
    """Feature-standardized MLP used as the smoke stand-in for F."""

    def __init__(self, n_in: int, n_out: int, width: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, n_out),
        )

    def fit_standardized(self, feat: np.ndarray, target: np.ndarray, train_mask: np.ndarray, steps: int, seed: int):
        torch.manual_seed(seed)
        mu = feat[train_mask].mean(0)
        sd = feat[train_mask].std(0)
        sd = np.where(sd < 1e-6, 1.0, sd)
        ym = target[train_mask].mean(0)
        ys = target[train_mask].std(0)
        ys = np.where(ys < 1e-6, 1.0, ys)
        self.mu = mu.astype(np.float32)
        self.sd = sd.astype(np.float32)
        self.ym = ym.astype(np.float32)
        self.ys = ys.astype(np.float32)
        xt = torch.tensor((feat[train_mask] - mu) / sd, dtype=torch.float32)
        yt = torch.tensor((target[train_mask] - ym) / ys, dtype=torch.float32)
        opt = torch.optim.Adam(self.parameters(), lr=2e-3)
        n = xt.shape[0]
        for _ in range(steps):
            sel = torch.randint(0, n, (1024,))
            opt.zero_grad()
            loss = (self.net(xt[sel]) - yt[sel]).pow(2).mean()
            loss.backward()
            opt.step()
        self.eval()

    @torch.no_grad()
    def predict(self, feat: np.ndarray) -> np.ndarray:
        z = (feat.astype(np.float32) - self.mu) / self.sd
        pred = self.net(torch.tensor(z, dtype=torch.float32)).numpy()
        return pred * self.ys + self.ym


def run_h1(packed: Dict[str, np.ndarray]) -> Dict:
    x = packed["x"]
    p_raw = packed["p"]
    a = packed["a"]
    g_next = packed["g_next"]
    seed = packed["seed_id"]
    # Hold out a few geometry seeds for the factual panel.
    test_seeds = {5, 18, 33}
    test = np.isin(seed, list(test_seeds))
    train = ~test
    # Drop non-finite rows.
    finite = np.isfinite(x).all(1) & np.isfinite(p_raw).all(1) & np.isfinite(g_next).all(1)
    train = train & finite
    test = test & finite

    p_torch = torch.tensor(p_raw, dtype=torch.float64)
    cleaned = sanitize_privileged(p_torch)
    p_causal = cleaned["p"].numpy()
    # Consumed stream must carry the lagged carrier in the future slice.
    consumed_violations = int(cleaned["consumed_violations"])

    def features(p):
        base = np.concatenate([x, p, a], axis=1)
        prod = x[:, :4] * a
        return np.concatenate([base, prod], axis=1).astype(np.float32)

    f_c = features(p_causal)
    f_l = features(p_raw)
    # cos, sin, qd, tip position, tip velocity x/y. Acceleration channels are
    # a separate diagnostic: a 5-point second difference is not a smooth map
    # of the same features, and it dominated the earlier full-vector score.
    kin = np.array([0, 1, 2, 4, 5, 6, 7, 8], dtype=np.int64)
    y_kin = g_next[:, kin].astype(np.float32)
    width = int(SCALE_FLAGS["h1_mlp_width"])
    steps = int(SCALE_FLAGS["h1_mlp_steps"])

    def fit_arm(feat, target, seed_i):
        model = _ScaledMLP(feat.shape[1], target.shape[1], width)
        model.fit_standardized(feat, target, train, steps, seed_i)
        pred = model.predict(feat[test])
        return _explained_variance(pred, target[test]), model

    ev_c, model_c = fit_arm(f_c, y_kin, 20261024)
    ev_l, model_l = fit_arm(f_l, y_kin, 20261024)
    ev_full_c, _ = fit_arm(f_c, g_next.astype(np.float32), 20261025)
    ev_full_l, _ = fit_arm(f_l, g_next.astype(np.float32), 20261025)
    retention = float(ev_c / ev_l) if ev_l > 0.5 else 0.0

    # Counterfactual: stale future slots, action column shifted, truth from a
    # resimulated plant. Both arms are the factual-panel MLPs.
    cf = _counterfactual_probe(packed, model_c, model_l, train)
    charter_pass = bool(retention > float(SCALE_FLAGS["h1_retention_min"]) and consumed_violations == 0 and ev_l > 0.5)
    return {
        "status": "Candidate",
        "metric_verdict": "PASS" if charter_pass else "FAIL",
        "pass_is_scale_conditional": True,
        "td_explained_variance_causal": ev_c,
        "td_explained_variance_leaked": ev_l,
        "td_explained_variance_full_vector_causal": ev_full_c,
        "td_explained_variance_full_vector_leaked": ev_full_l,
        "target": str(SCALE_FLAGS["h1_target"]),
        "retention": retention,
        "retention_threshold": float(SCALE_FLAGS["h1_retention_min"]),
        "consumed_causality_violations": consumed_violations,
        "raw_fail_closed_rows": int(cleaned["raw_fail_closed_rows"]),
        "freeze_normalizer_on_raw_future": bool(cleaned["freeze_normalizer"]),
        "n_train": int(train.sum()),
        "n_test": int(test.sum()),
        "counterfactual": cf,
        "p_dim_executed": P_DIM,
        "p_dim_charter": 5065,
    }


def _counterfactual_probe(packed, model_c, model_l, train_mask) -> Dict[str, float]:
    """Replay 48 E3 rows one control step with a perturbed torque.

    Leaked features keep the original future slice (stale). Causal features do not
    contain that slice. Compare prediction error against the resimulated outcome.
    """
    scene = packed["scene_id"]
    idx = np.flatnonzero((scene == 3) & train_mask)
    if idx.size < 48:
        idx = np.flatnonzero(train_mask)
    rng = np.random.default_rng(1)
    choose = rng.choice(idx, size=min(48, idx.size), replace=False)

    # Recover plant state from packed g and the stored scalars. g has cos/sin/qd.
    g = packed["g"][choose]
    q = np.arctan2(g[:, 1], g[:, 0])
    qd = g[:, 2]
    length = packed["length"][choose]
    mass = packed["mass"][choose]
    s_max = packed["s_max"][choose]
    # Inertia is inside x column 9 / 50.
    inertia = packed["x"][choose, 9] / 50.0
    tau = packed["tau_next"][choose]
    u0 = packed["a"][choose, 0]
    u1 = u0 + 0.35
    dt = float(SCALE_FLAGS["coverage_physics_dt_s"])
    n_sub = int(SCALE_FLAGS["coverage_substeps_per_control"])

    def resim(u_np):
        q_t = torch.tensor(q, dtype=torch.float64)
        qd_t = torch.tensor(qd, dtype=torch.float64)
        u_t = torch.tensor(u_np, dtype=torch.float64)
        tau_t = torch.tensor(tau, dtype=torch.float64)
        br = torch.zeros_like(q_t)
        m = torch.tensor(mass, dtype=torch.float64)
        L = torch.tensor(length, dtype=torch.float64)
        J = torch.tensor(inertia, dtype=torch.float64).clamp_min(1e-5)
        qmin = torch.full_like(q_t, -2.0)
        qmax = torch.full_like(q_t, 2.0)
        qdd = torch.zeros_like(q_t)
        for _ in range(n_sub):
            q_t, qd_t, br, qdd, _, _, _ = single_joint_substep(
                q_t, qd_t, u_t, tau_t, br, m, L, J, qmin, qmax, dt
            )
        return q_t, qd_t, qdd

    q_n, qd_n, _qdd_n = resim(u1)
    # First three kinematic outputs are cos, sin, qd.
    truth = torch.stack([torch.cos(q_n), torch.sin(q_n), qd_n], dim=-1).numpy()

    def perturbed(p_rows, a_rows):
        x_rows = packed["x"][choose]
        prod = x_rows[:, :4] * a_rows
        return np.concatenate([x_rows, p_rows, a_rows, prod], axis=1).astype(np.float32)

    p_c = sanitize_privileged(torch.tensor(packed["p"][choose], dtype=torch.float64))["p"].numpy()
    p_l = packed["p"][choose].copy()  # stale future from the unperturbed transition
    a_rows = u1.reshape(-1, 1)
    pred_c = model_c.predict(perturbed(p_c, a_rows))[:, :3]
    pred_l = model_l.predict(perturbed(p_l, a_rows))[:, :3]
    err_c = float(np.mean((pred_c - truth) ** 2))
    err_l = float(np.mean((pred_l - truth) ** 2))
    return {
        "counterfactual_mse_causal": err_c,
        "counterfactual_mse_leaked_stale_future": err_l,
        "causal_better": bool(err_c < err_l),
    }


def run_h2(roll: Dict[str, np.ndarray], panel: Dict[str, float]) -> Dict:
    T, B = roll["qd"].shape
    n_cfg = 144
    cells = np.zeros((n_cfg, 5), dtype=np.int64)
    cfg = roll["config_id"]
    scene = roll["scene_id"]
    for s in range(5):
        for c in range(n_cfg):
            # one episode per cell, T steps, if finite
            cells[c, s] = T
    finite_eps = roll["finite"]
    if not bool(finite_eps.all()):
        bad = np.flatnonzero(~finite_eps)
        for b in bad:
            cells[int(cfg[b]), int(scene[b])] = 0

    e3 = scene == 3
    e4 = scene == 4
    e1 = scene == 1
    hold = 0
    recovery = 0
    e3_sched_ok = True
    e4_sched_ok = True
    for b in np.flatnonzero(e3):
        onset = int(roll["e3_onset"][b])
        length = int(roll["e3_len"][b])
        if length < 25 or onset + length > T:
            e3_sched_ok = False
            continue
        window = roll["qd"][onset : onset + length, b]
        # Settled portion of the sustained window.
        settled = window[-15:]
        if np.mean(np.abs(settled)) < 0.35:
            hold += 1
    for b in np.flatnonzero(e4):
        rel = int(roll["e4_release"][b])
        if rel >= T or (T - rel) < 20:
            e4_sched_ok = False
            continue
        # A real release: load before the edge, ~0 after.
        before = np.max(np.abs(roll["tau_cmd"][b, :rel])) if rel > 0 else 0.0
        after = np.max(np.abs(roll["tau_cmd"][b, rel:]))
        if before < 1e-6 or after > 1e-6:
            e4_sched_ok = False
        tail = roll["qd"][-10:, b]
        if np.mean(tail ** 2) ** 0.5 < 0.35:
            recovery += 1

    # λ = 0.92 witness: a finite step inside the reserve with the actuator working.
    witness = np.zeros(n_cfg, dtype=bool)
    u = roll["u"]
    s_max = roll["s_max"]
    for b in range(B):
        util = np.abs(u[:, b]) / max(float(s_max[b]), 1e-6)
        if np.any((util <= 0.92 + 1e-4) & (util >= 0.15) & np.isfinite(u[:, b])):
            witness[int(cfg[b])] = True

    # E1 escape: started near the stop (qmax - q small at step 0) and ended clear.
    escapes = 0
    e1_n = int(e1.sum())
    for b in np.flatnonzero(e1):
        # E1 is initialized on the positive stop before warmup. Success is a
        # final pose clear of that stop with bounded speed, not a still-contacting joint.
        gap1 = float(roll["qmax"][b] - roll["q"][-1, b])
        if gap1 > 0.20 and abs(float(roll["qd"][-1, b])) < 1.5:
            escapes += 1
    escape_rate = float(escapes / max(e1_n, 1))

    # Distinct anchors across the config axis (de-synchronization happened).
    anchor_nunique = int(len(np.unique(roll["anchor"])))

    coverage_ready = bool(
        cells.min() >= 1
        and e3_sched_ok
        and e4_sched_ok
        and bool(witness.all())
        and hold >= n_cfg
        and recovery >= n_cfg
        and panel["zero_vel_rmse_worst"] < float(SCALE_FLAGS["zero_vel_rmse_max"])
        and panel["control_fail_count"] == 0
        and panel["negative_control_position_abs_err"] > 0.30
        and roll["infeasible_programs"] == 0
        and escape_rate >= 0.9
        and bool(finite_eps.all())
    )
    return {
        "status": "Candidate",
        "metric_verdict": "PASS" if coverage_ready else "FAIL",
        "pass_is_scale_conditional": True,
        "coverage_ready": coverage_ready,
        "joint_cells": int(cells.size),
        "joint_cell_min_steps": int(cells.min()),
        "joint_cell_max_steps": int(cells.max()),
        "marginal_scene_steps": cells.sum(0).tolist(),
        "marginals_do_not_certify_joint": True,
        "sustained_load_hold_count": int(hold),
        "post_load_recovery_count": int(recovery),
        "e3_schedule_ok": e3_sched_ok,
        "e4_schedule_ok": e4_sched_ok,
        "lambda_092_witness_configs": int(witness.sum()),
        "limit_escape_rate": escape_rate,
        "anchor_levels_observed": anchor_nunique,
        "infeasible_programs": int(roll["infeasible_programs"]),
        "zero_vel_rmse_mean": panel["zero_vel_rmse_mean"],
        "zero_vel_rmse_worst": panel["zero_vel_rmse_worst"],
        "zero_vel_rmse_threshold": float(SCALE_FLAGS["zero_vel_rmse_max"]),
        "hold_position_abs_err_worst": panel["hold_position_abs_err_worst"],
        "control_fail_count": panel["control_fail_count"],
        "negative_control_rmse": panel["negative_control_rmse"],
        "negative_control_position_abs_err": panel["negative_control_position_abs_err"],
        "executed_mixture": "uniform_one_episode_per_cell",
        "charter_mixture": "E0:0.20,E1:0.15,E2:0.20,E3:0.15,E4:0.30",
    }


def run_h3(packed: Dict[str, np.ndarray], device: torch.device) -> Dict:
    torch.manual_seed(20261024)
    g = torch.tensor(packed["g"], dtype=torch.float32, device=device)
    g_next = torch.tensor(packed["g_next"], dtype=torch.float32, device=device)
    # Schema: repair tiny drift of cos/sin from packing noise by renormalizing nothing;
    # values come from cos/sin directly so the guard should pass.
    if schema_guard_g(g[:64]) != "OK":
        return {
            "status": "Candidate",
            "metric_verdict": "FAIL",
            "pass_is_scale_conditional": True,
            "reason": "SCHEMA_INCOMPATIBLE",
        }
    rho = torch.tensor(packed["length"], dtype=torch.float32, device=device)
    seed = packed["seed_id"].astype(np.int64)
    hold = set(int(s) for s in SCALE_FLAGS["h3_holdout_geometry_seeds"])
    train_m = torch.tensor([int(s) not in hold for s in seed], device=device)
    test_m = ~train_m
    g_hat = t_kappa(g, rho)
    g_next_hat = t_kappa(g_next, rho)
    a = torch.tensor(packed["a"], dtype=torch.float32, device=device)
    a_n = a / torch.tensor(packed["s_max"], dtype=torch.float32, device=device).unsqueeze(-1).clamp_min(1e-6)
    lam = g_hat[:, 9:10]
    geom = torch.stack(
        [
            rho / 0.22,
            torch.tensor(packed["mass"], dtype=torch.float32, device=device) / 0.55,
            torch.tensor(packed["c0"], dtype=torch.float32, device=device) / 6.0,
            torch.tensor(packed["amp"], dtype=torch.float32, device=device),
            torch.tensor(packed["freq"], dtype=torch.float32, device=device),
            lam.squeeze(-1),
        ],
        dim=-1,
    )
    cfg = torch.tensor(packed["config_id"].astype(np.int64), device=device)

    class BMap(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(13, 64), nn.SiLU(),
                nn.Linear(64, 16),
            )
            # Nonlinear decoder so a curved code can still reconstruct g.
            self.dec = nn.Sequential(
                nn.Linear(16, 64), nn.SiLU(),
                nn.Linear(64, 13),
            )

        def forward(self, g_in):
            return project_to_sphere(self.net(g_in), 4.0)

    class FMap(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(13 + 1 + 6, int(SCALE_FLAGS["h3_width"])),
                nn.SiLU(),
                nn.Linear(int(SCALE_FLAGS["h3_width"]), int(SCALE_FLAGS["h3_width"])),
                nn.SiLU(),
                nn.Linear(int(SCALE_FLAGS["h3_width"]), 16),
            )

        def forward(self, g_in, a_in, geom_in):
            return self.net(torch.cat([g_in, a_in, geom_in], dim=-1))

    # Per-dimension scale so a large acceleration channel cannot hide a collapsed code.
    mu = g_hat[train_m].mean(0)
    sd = g_hat[train_m].std(0).clamp_min(1e-3)
    g_s = (g_hat - mu) / sd
    g_next_s = (g_next_hat - mu) / sd

    bmap = BMap().to(device)
    fmap = FMap().to(device)
    opt_b = torch.optim.Adam(bmap.parameters(), lr=2e-3)
    train_idx = torch.nonzero(train_m, as_tuple=False).squeeze(-1)
    # Counts per train config for the support sampler.
    train_cfgs = cfg[train_idx].detach().cpu().numpy()
    counts: Dict[int, int] = {}
    for c in train_cfgs.tolist():
        counts[int(c)] = counts.get(int(c), 0) + 1
    sampler = SupportSampler(counts)
    present = [c for c, n in counts.items() if n >= 2]
    draw = sampler.sample(present[:8] if len(present) >= 8 else present, mixture=0.1)
    insufficient = sampler.sample(present[:4] + [-999], mixture=0.1)

    erank_floor = float(SCALE_FLAGS["h3_erank_floor"])

    def erank_hinge(z_group: torch.Tensor) -> torch.Tensor:
        zc = z_group - z_group.mean(0, keepdim=True)
        cov = (zc.transpose(0, 1) @ zc) / float(z_group.shape[0])
        ev = torch.linalg.eigvalsh(cov).clamp_min(1e-8)
        p = ev / ev.sum()
        er = torch.exp(-(p * torch.log(p)).sum())
        # Per-config effective-rank floor. Not a blind covariance-to-identity penalty.
        return torch.relu(erank_floor - er)

    steps = int(SCALE_FLAGS["h3_train_steps"])
    bs = int(SCALE_FLAGS["h3_batch"])
    for step in range(steps):
        sel = train_idx[torch.randint(0, train_idx.numel(), (bs,), device=device)]
        opt_b.zero_grad()
        z = bmap(g_s[sel])
        rec = (bmap.dec(z) - g_s[sel]).pow(2).mean()
        # Stratified hinge on four configs inside the batch.
        hinge = torch.zeros((), device=device)
        uniq = cfg[sel].unique()
        used = 0
        for c in uniq[:8]:
            mask = cfg[sel] == c
            if int(mask.sum()) >= 8:
                hinge = hinge + erank_hinge(z[mask])
                used += 1
        if used:
            hinge = hinge / used
        loss = rec + 0.15 * hinge
        loss.backward()
        opt_b.step()

    bmap.eval()
    for p in bmap.parameters():
        p.requires_grad_(False)
    opt_f = torch.optim.Adam(fmap.parameters(), lr=1.5e-3)
    for step in range(steps):
        # Hierarchical draw: pick configs, then rows inside them.
        pick = [present[int(i)] for i in torch.randint(0, len(present), (8,))]
        decision = sampler.sample(pick, mixture=0.1)
        if decision["status"] != "OK":
            continue
        parts = []
        for c in pick:
            pool = train_idx[cfg[train_idx] == c]
            if pool.numel() == 0:
                continue
            take = pool[torch.randint(0, pool.numel(), (64,), device=device)]
            parts.append(take)
        if not parts:
            continue
        sel = torch.cat(parts, 0)
        opt_f.zero_grad()
        with torch.no_grad():
            target = bmap(g_next_s[sel])
        pred = fmap(g_s[sel], a_n[sel], geom[sel])
        # Configuration-matched successor. No pooled-shell target.
        loss = (pred - target).pow(2).mean()
        loss.backward()
        opt_f.step()

    def group_rel(mask: torch.Tensor) -> Dict[str, float]:
        if int(mask.sum()) == 0:
            return {"worst": None, "mean": None, "n_groups": 0}
        with torch.no_grad():
            target = bmap(g_next_s[mask])
            pred = fmap(g_s[mask], a_n[mask], geom[mask])
            rel = (pred - target).norm(dim=-1) / target.norm(dim=-1).clamp_min(1e-6)
            cfgs = cfg[mask]
            worst = 0.0
            total = 0.0
            n_groups = 0
            pooled_err = []
            for c in cfgs.unique().tolist():
                m = cfgs == c
                val = float(rel[m].mean().item())
                worst = max(worst, val)
                total += val
                n_groups += 1
                # Pooled reference: distance to a shuffled other-config embedding.
            perm = torch.randperm(int(mask.sum()), device=device)
            pooled = (pred - target[perm]).norm(dim=-1) / target[perm].norm(dim=-1).clamp_min(1e-6)
            pooled_mean = float(pooled.mean().item())
        return {
            "worst": worst,
            "mean": total / max(n_groups, 1),
            "n_groups": n_groups,
            "pooled_reference_mean_rel": pooled_mean,
        }

    hold_stats = group_rel(test_m)
    train_stats = group_rel(train_m)
    # Per-config erank on a slice of training data, after freeze.
    eranks = []
    with torch.no_grad():
        z_all = bmap(g_s[train_m])
        cfgs = cfg[train_m]
        for c in cfgs.unique()[::6].tolist():
            zc = z_all[cfgs == c]
            if zc.shape[0] >= 16:
                eranks.append(participation_ratio(zc))
    worst_erank = float(min(eranks)) if eranks else float("nan")
    # Tip length-relative probe: linear map from z to tip xy / rho on holdout.
    with torch.no_grad():
        z_te = bmap(g_s[test_m])
        tip = g[test_m][:, 4:6] / rho[test_m].unsqueeze(-1).clamp_min(1e-6)
        # lstsq
        sol = torch.linalg.lstsq(z_te, tip).solution
        tip_rmse = float((z_te @ sol - tip).pow(2).mean().sqrt().item())
    threshold = float(SCALE_FLAGS["h3_worst_group_rel_err_max"])
    tip_max = float(SCALE_FLAGS["h3_tip_rmse_max"])
    worst = hold_stats["worst"]
    # A constant code on the sphere has norm 4, so relative successor error
    # can look small while the representation has collapsed. Rank is part of the gate.
    passed = (
        worst is not None
        and worst < threshold
        and worst_erank >= erank_floor
        and tip_rmse < tip_max
        and insufficient["status"] == "INSUFFICIENT_SUPPORT"
    )
    return {
        "status": "Candidate",
        "metric_verdict": "PASS" if passed else "FAIL",
        "pass_is_scale_conditional": True,
        "worst_group_relative_successor_error_holdout": worst,
        "mean_group_relative_successor_error_holdout": hold_stats["mean"],
        "pooled_reference_mean_rel_holdout": hold_stats.get("pooled_reference_mean_rel"),
        "worst_group_relative_successor_error_train": train_stats["worst"],
        "threshold": threshold,
        "n_holdout_groups": hold_stats["n_groups"],
        "n_train_groups": train_stats["n_groups"],
        "holdout_geometry_seeds": list(SCALE_FLAGS["h3_holdout_geometry_seeds"]),
        "worst_stratified_erank_train_subsample": worst_erank,
        "erank_floor": erank_floor,
        "charter_erank_floor": float(SCALE_FLAGS["h3_charter_erank_floor"]),
        "tip_length_relative_rmse_holdout": tip_rmse,
        "ipw_missing_stratum_status": insufficient["status"],
        "matched_reference": "same_config_successor",
        "t_kappa_isolated_from_input_normalizer": True,
    }


class _Upper(nn.Module):
    def __init__(self, d_in: int, d_out: int, width: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, width),
            nn.LayerNorm(width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.LayerNorm(width),
            nn.SiLU(),
            nn.Linear(width, d_out),
        )
        nn.init.normal_(self.net[-1].weight, std=0.08)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, obs):
        return self.net(obs)


def _limb_constants(device, dtype):
    return {
        "m1": float(SCALE_FLAGS["limb_m1_kg"]),
        "m2": float(SCALE_FLAGS["limb_m2_kg"]),
        "l1": float(SCALE_FLAGS["limb_l1_m"]),
        "m_payload": float(SCALE_FLAGS["limb_m_payload_kg"]),
        "q_target": torch.tensor(SCALE_FLAGS["limb_q_target"], device=device, dtype=dtype),
    }


def _classify_limb_cells(device, dtype):
    c = _limb_constants(device, dtype)
    cells = []
    grid = [(L2, C02, "grid") for L2 in SCALE_FLAGS["grid_L2_m"] for C02 in SCALE_FLAGS["grid_C02_nm"]]
    grid.append((SCALE_FLAGS["stress_L2_m"], SCALE_FLAGS["stress_C02_nm"], "stress"))
    q1 = c["q_target"][0]
    q2 = c["q_target"][1]
    for L2, C02, panel in grid:
        u1, u2 = static_required_torque(q1, q2, c["m1"], c["m2"], c["l1"], float(L2), c["m_payload"])
        s1 = 1.25 * float(SCALE_FLAGS["grid_C01_nm"])
        s2 = 1.25 * float(C02)
        d1 = abs(float(u1)) / s1
        d2 = abs(float(u2)) / s2
        demand = max(d1, d2)
        if demand <= float(SCALE_FLAGS["edge_reserve"]):
            label = "IN"
        elif demand <= float(SCALE_FLAGS["publish_reserve_lambda"]):
            label = "EDGE"
        else:
            label = "OUT"
        q2t = torch.tensor([float(q2)], device=device, dtype=dtype)
        m11, _, m22, _ = limb_mass_matrix(q2t, c["m1"], c["m2"], c["l1"], float(L2))
        cells.append(
            {
                "L2": float(L2),
                "C02": float(C02),
                "C01": float(SCALE_FLAGS["grid_C01_nm"]),
                "S1": s1,
                "S2": s2,
                "panel": panel,
                "demand": demand,
                "label": label,
                "feasible": label in ("IN", "EDGE"),
                "J1": float(m11.item()),
                "J2": float(m22.item()),
                "u_required": [float(u1), float(u2)],
            }
        )
    return cells


def _own_gravity(q, m1, m2, l1, l2):
    lc1 = 0.5 * l1
    lc2 = 0.5 * l2
    g1 = -9.81 * m1 * lc1 * torch.sin(q[:, 0])
    g2 = -9.81 * m2 * lc2 * torch.sin(q[:, 1])
    return torch.stack([g1, g2], dim=-1)


def _decode_setpoint(q, z, S):
    """Absolute joint setpoint and feedforward torque held for one control step.

    The latent trim is a residual about the measured angle. A neutral code
    therefore asks the local servo to stay, and the feedforward channel has
    to carry the pelvis load the own-link gravity term does not.
    """
    delta = FROZEN_GAINS["h4_q_scale"] * torch.tanh(z[:, 0:2] / 2.0)
    q_des = q.detach() + delta
    tau_ff = FROZEN_GAINS["h4_ff_scale"] * S * torch.tanh(z[:, 2:4] / 2.0)
    return q_des, tau_ff


def _torque_servo(q, qd, q_des, tau_ff, S, g_own):
    kp = FROZEN_GAINS["h4_kp"]
    kd = FROZEN_GAINS["h4_kd"]
    u = g_own + tau_ff + kp * (q_des - q) - kd * qd
    return torch.clamp(u, -0.92 * S, 0.92 * S)


def _upper_obs(q, qd, u_prev, q_target, L2, S, z_hip, zd_hip):
    return torch.cat(
        [
            q,
            qd,
            torch.sin(q),
            torch.cos(q),
            q_target.expand_as(q),
            L2.unsqueeze(-1),
            S / 10.0,
            u_prev / 10.0,
            z_hip.unsqueeze(-1),
            zd_hip.unsqueeze(-1),
        ],
        dim=-1,
    )


def _initial_q(q_target: torch.Tensor) -> torch.Tensor:
    offset = torch.tensor(
        SCALE_FLAGS["h4_init_offset_rad"], device=q_target.device, dtype=q_target.dtype
    )
    return q_target + offset


def _substep_torque(mode, q, qd, q_des, tau_ff, batch, cnst):
    damp = float(SCALE_FLAGS["h4_learning_damp"])
    dt = float(SCALE_FLAGS["h4_physics_dt_s"])
    n_sub = int(SCALE_FLAGS["h4_substeps_per_control"])
    for _ in range(n_sub):
        if mode in ("A", "A-rand"):
            g_own = _own_gravity(q, cnst["m1"], cnst["m2"], cnst["l1"], batch["L2"])
            u = _torque_servo(q, qd, q_des, tau_ff, batch["S"], g_own)
        elif mode == "C":
            u = tau_ff
        elif mode == "oracle":
            g1, g2 = limb_gravity(
                cnst["q_target"][0], cnst["q_target"][1], cnst["m1"], cnst["m2"], cnst["l1"], batch["L2"]
            )
            p1, p2 = pelvis_load_torque(
                cnst["q_target"][0], cnst["q_target"][1], cnst["l1"], batch["L2"], cnst["m_payload"]
            )
            u_hold = torch.stack([g1 - p1, g2 - p2], dim=-1)
            u = _torque_servo(q, qd, q_des, u_hold, batch["S"], torch.zeros_like(q))
        else:
            raise KeyError(mode)
        q, qd = limb_substep(
            q, qd, u, cnst["m1"], cnst["m2"], cnst["l1"], batch["L2"], cnst["m_payload"], dt, damp=damp
        )
    return q, qd, u


def _rollout_cost(mode: str, policy: nn.Module, batch: Dict[str, torch.Tensor], cnst: Dict, train: bool):
    T = int(SCALE_FLAGS["h4_episode_steps"])
    B = batch["L2"].shape[0]
    q_target = cnst["q_target"].unsqueeze(0).expand(B, -1)
    q = _initial_q(q_target)
    qd = torch.zeros_like(q)
    u_prev = torch.zeros_like(q)
    loss_task = torch.zeros((), device=q.device, dtype=q.dtype)
    hinge = torch.zeros((), device=q.device, dtype=q.dtype)
    z_hip_prev = None
    for t in range(T):
        _x, zhip = forward_kinematics(q[:, 0], q[:, 1], cnst["l1"], batch["L2"])
        zd = torch.zeros_like(zhip) if z_hip_prev is None else (zhip - z_hip_prev) / float(SCALE_FLAGS["control_dt_s"])
        z_hip_prev = zhip.detach()
        obs = _upper_obs(q, qd, u_prev, q_target, batch["L2"], batch["S"], zhip, zd)
        logit = policy(obs)
        hinge = hinge + torch.relu(logit.abs() - 8.0).mean()
        if mode == "A":
            zlat = project_to_sphere(logit, 4.0)
            q_des, tau_ff = _decode_setpoint(q, zlat, batch["S"])
        elif mode == "C":
            q_des = q_target
            tau_ff = 0.92 * batch["S"] * torch.tanh(logit[:, 0:2])
        else:
            raise KeyError(mode)
        q, qd, u = _substep_torque(mode, q, qd, q_des, tau_ff, batch, cnst)
        u_prev = u.detach()
        loss_task = loss_task + (q - q_target).pow(2).mean() + 0.01 * qd.pow(2).mean()
    loss = loss_task / T + 1e-3 * hinge / T
    return loss


def _eval_cells(mode, policy, batch, cnst) -> Dict[str, float]:
    T = int(SCALE_FLAGS["h4_episode_steps"])
    B = batch["L2"].shape[0]
    q_target = cnst["q_target"].unsqueeze(0).expand(B, -1)
    q = _initial_q(q_target)
    qd = torch.zeros_like(q)
    u_prev = torch.zeros_like(q)
    z_hip_prev = None
    pos_acc = torch.zeros(B, device=q.device, dtype=q.dtype)
    vel_acc = torch.zeros_like(pos_acc)
    n_tail = 0
    with torch.no_grad():
        for t in range(T):
            _x, zhip = forward_kinematics(q[:, 0], q[:, 1], cnst["l1"], batch["L2"])
            zd = torch.zeros_like(zhip) if z_hip_prev is None else (zhip - z_hip_prev) / float(SCALE_FLAGS["control_dt_s"])
            z_hip_prev = zhip
            obs = _upper_obs(q, qd, u_prev, q_target, batch["L2"], batch["S"], zhip, zd)
            if mode == "A":
                logit = policy(obs)
                zlat = project_to_sphere(logit, 4.0)
                q_des, tau_ff = _decode_setpoint(q, zlat, batch["S"])
            elif mode == "C":
                logit = policy(obs)
                q_des = q_target
                tau_ff = 0.92 * batch["S"] * torch.tanh(logit[:, 0:2])
            elif mode == "oracle":
                q_des = q_target
                tau_ff = torch.zeros_like(q)
            else:
                zlat = project_to_sphere(torch.randn(B, 16, device=q.device, dtype=q.dtype), 4.0)
                q_des, tau_ff = _decode_setpoint(q, zlat, batch["S"])
                mode = "A-rand"
            q, qd, u = _substep_torque(mode, q, qd, q_des, tau_ff, batch, cnst)
            u_prev = u
            if t >= T // 2:
                pos_acc = pos_acc + (q - q_target).pow(2).mean(-1).sqrt()
                vel_acc = vel_acc + qd.pow(2).mean(-1).sqrt()
                n_tail += 1
    pos = pos_acc / max(n_tail, 1)
    vel = vel_acc / max(n_tail, 1)
    return {
        "per_cell_pos_rmse": [float(v) for v in pos.detach().cpu()],
        "per_cell_vel_rmse": [float(v) for v in vel.detach().cpu()],
        "worst_pos_rmse": float(pos.max().item()),
        "worst_vel_rmse": float(vel.max().item()),
        "mean_pos_rmse": float(pos.mean().item()),
    }


def run_h4(device: torch.device) -> Dict:
    torch.manual_seed(20261024)
    dtype = torch.float32
    cells = _classify_limb_cells(device, dtype)
    learn_cells = [c for c in cells if c["feasible"] and c["panel"] == "grid"]
    if not learn_cells:
        return {
            "status": "Candidate",
            "metric_verdict": "FAIL",
            "pass_is_scale_conditional": True,
            "reason": "no_feasible_grid_cells",
            "cells": cells,
        }
    cnst = _limb_constants(device, dtype)
    batch = {
        "L2": torch.tensor([c["L2"] for c in learn_cells], device=device, dtype=dtype),
        "S": torch.tensor([[c["S1"], c["S2"]] for c in learn_cells], device=device, dtype=dtype),
        "J": torch.tensor([[c["J1"], c["J2"]] for c in learn_cells], device=device, dtype=dtype),
    }
    d_in = _upper_obs(
        torch.zeros(1, 2, device=device),
        torch.zeros(1, 2, device=device),
        torch.zeros(1, 2, device=device),
        cnst["q_target"].unsqueeze(0),
        batch["L2"][:1],
        batch["S"][:1],
        torch.zeros(1, device=device),
        torch.zeros(1, device=device),
    ).shape[-1]
    width = int(SCALE_FLAGS["h4_upper_width"])
    pol_a = _Upper(d_in, 16, width).to(device)
    pol_c = _Upper(d_in, 16, width).to(device)
    # Sphere calibration: unused latent axes carry the norm so the decoded
    # trim and feedforward start near zero. Flat torque keeps a zero bias.
    with torch.no_grad():
        pol_a.net[-1].bias[4:].fill_(1.5)
    n_a = sum(p.numel() for p in pol_a.parameters())
    n_c = sum(p.numel() for p in pol_c.parameters())
    assert n_a == n_c
    hash_before = frozen_local_hash(FROZEN_GAINS)
    opt_a = torch.optim.Adam(pol_a.parameters(), lr=3e-3)
    opt_c = torch.optim.Adam(pol_c.parameters(), lr=3e-3)
    updates = int(SCALE_FLAGS["h4_updates"])
    every = int(SCALE_FLAGS["h4_log_every"])
    curve_a = []
    curve_c = []

    def log(policy, mode, update):
        stats = _eval_cells(mode, policy, batch, cnst)
        stats["update"] = update
        return stats

    curve_a.append(log(pol_a, "A", 0))
    curve_c.append(log(pol_c, "C", 0))
    for update in range(1, updates + 1):
        opt_a.zero_grad()
        loss_a = _rollout_cost("A", pol_a, batch, cnst, True)
        loss_a.backward()
        torch.nn.utils.clip_grad_norm_(pol_a.parameters(), 1.0)
        opt_a.step()
        opt_c.zero_grad()
        loss_c = _rollout_cost("C", pol_c, batch, cnst, True)
        loss_c.backward()
        torch.nn.utils.clip_grad_norm_(pol_c.parameters(), 1.0)
        opt_c.step()
        if update % every == 0 or update == updates:
            curve_a.append(log(pol_a, "A", update))
            curve_c.append(log(pol_c, "C", update))
    hash_after = frozen_local_hash(FROZEN_GAINS)
    oracle = _eval_cells("oracle", pol_a, batch, cnst)
    arand = _eval_cells("A-rand", pol_a, batch, cnst)

    def auc(curve):
        return float(np.mean([row["worst_pos_rmse"] for row in curve]))

    auc_a = auc(curve_a)
    auc_c = auc(curve_c)
    final_a = curve_a[-1]["worst_pos_rmse"]
    final_c = curve_c[-1]["worst_pos_rmse"]

    def first_hit(curve):
        for row in curve:
            if row["worst_pos_rmse"] < 0.20 and row["worst_vel_rmse"] < float(SCALE_FLAGS["zero_vel_rmse_max"]):
                return int(row["update"])
        return None

    # Positive skill on the nominal cell (L2=0.25, C02=6) versus the worst cell.
    def skill(row):
        # R̃ = exp(-rmse / sigma)
        return [math.exp(-v / 0.25) for v in row["per_cell_pos_rmse"]]

    # Nominal index
    nominal = 0
    for i, c in enumerate(learn_cells):
        if abs(c["L2"] - 0.25) < 1e-9 and abs(c["C02"] - 6.0) < 1e-9:
            nominal = i
    skill_a = skill(curve_a[-1])
    skill_c = skill(curve_c[-1])
    delta_a = [s - skill_a[nominal] for s in skill_a]
    delta_c = [s - skill_c[nominal] for s in skill_c]
    contrast = [da - dc for da, dc in zip(delta_a, delta_c)]

    # Five-code diagnostic at the nominal operating point, not a certificate.
    with torch.no_grad():
        q_t = cnst["q_target"]
        u1, u2 = static_required_torque(
            q_t[0], q_t[1], cnst["m1"], cnst["m2"], cnst["l1"], learn_cells[nominal]["L2"], cnst["m_payload"]
        )
        smax = max(learn_cells[nominal]["S1"], learn_cells[nominal]["S2"])
        codes = {
            "A-AMP": abs(float(u2)) / (0.92 * learn_cells[nominal]["S2"]),
            "A-BW": 1.0,  # hold task, no commanded oscillation
            "A-IMP": 0.0,
            "A-RHO": learn_cells[nominal]["demand"],
            "A-NULL": abs(float(u1) + float(u2)) * 0.0 + abs(
                float(u1) - learn_cells[nominal]["u_required"][0]
            ),
            "certifies_h4": False,
            "note": "Grade-2 diagnostic at the nominal static operating point. Amplitude inclusion is not coverage.",
        }

    passed = bool(
        auc_a < 0.85 * auc_c
        and final_a <= final_c + 0.02
        and oracle["worst_pos_rmse"] < 0.15
        and hash_before == hash_after
        and n_a == n_c
    )
    return {
        "status": "Candidate",
        "metric_verdict": "PASS" if passed else "FAIL",
        "pass_is_scale_conditional": True,
        "definition": (
            "PASS when Mode A worst-cell position-RMSE area under the learning curve "
            "is below 85% of equal-parameter flat-torque Baseline C, Mode A is not worse "
            "at the final update, the gravity/payload oracle holds every feasible cell "
            "(worst tail position RMSE below 0.15) from the registered pose offset, "
            "and the frozen-local hash is unchanged. Episodes start at the target plus "
            "that offset. The local servo is a torque PD toward a latent setpoint."
        ),
        "auc_worst_pos_rmse_mode_A": auc_a,
        "auc_worst_pos_rmse_baseline_C": auc_c,
        "final_worst_pos_rmse_mode_A": final_a,
        "final_worst_pos_rmse_baseline_C": final_c,
        "updates_to_pos0.20_vel0.35_mode_A": first_hit(curve_a),
        "updates_to_pos0.20_vel0.35_baseline_C": first_hit(curve_c),
        "equal_trainable_params": n_a,
        "charter_upper_trainable_params": 140000,
        "frozen_hash_before": hash_before,
        "frozen_hash_after": hash_after,
        "frozen_hash_unchanged": hash_before == hash_after,
        "oracle_worst_pos_rmse": oracle["worst_pos_rmse"],
        "a_rand_worst_pos_rmse": arand["worst_pos_rmse"],
        "curve_mode_A": curve_a,
        "curve_baseline_C": curve_c,
        "delta_skill_A_minus_nominal": delta_a,
        "delta_skill_C_minus_nominal": delta_c,
        "paired_contrast_A_minus_C": contrast,
        "worst_cell_paired_contrast": float(min(contrast)) if contrast else None,
        "cells": cells,
        "learn_cell_count": len(learn_cells),
        "five_code_diagnostic": codes,
        "rail_in_upper_obs_only": True,
        "mode_B": "not_executed",
        "mode_B_note": "Smooth IK adapter is a Grade-2 diagnostic and is not trained in this smoke.",
    }


def run_all() -> Dict:
    t0 = time.time()
    device = _device()
    closure = spec_closure()
    if not closure["pass"]:
        # Still return a structured failure so the notebook writes JSON.
        return {
            "program": "motor-unit-representation",
            "stage": 2,
            "scientific_lock": False,
            "lock_reason": "Specification closure failed. No hypothesis is locked.",
            "scale_flags": SCALE_FLAGS,
            "spec_closure": closure,
            "hypotheses": {},
            "device": str(device),
            "elapsed_sec": round(time.time() - t0, 4),
        }
    configs = build_unit_configs()
    roll = rollout_coverage(configs)
    panel = static_hold_panel(configs)
    packed = _build_step_tensors(roll)
    h1 = run_h1(packed)
    h2 = run_h2(roll, panel)
    h3 = run_h3(packed, device)
    h4 = run_h4(device)
    hash_end = frozen_local_hash(FROZEN_GAINS)
    summary = {
        "program": "motor-unit-representation",
        "stage": 2,
        "scientific_lock": False,
        "lock_reason": (
            "Smoke-scale execution. Privileged width, proprioceptive history, "
            "network width, upper-parameter budget, and the learning timestep "
            "differ from the charter protocol. Numeric PASS values do not lock H1–H4."
        ),
        "hypothesis_status": "Candidate",
        "scale_flags": SCALE_FLAGS,
        "spec_closure": closure,
        "hypotheses": {"H1": h1, "H2": h2, "H3": h3, "H4": h4},
        "metric_verdicts": {
            "H1": h1["metric_verdict"],
            "H2": h2["metric_verdict"],
            "H3": h3["metric_verdict"],
            "H4": h4["metric_verdict"],
        },
        "frozen_hash_end": hash_end,
        "frozen_hash_matches_spec": hash_end == closure["frozen_hash"],
        "device": str(device),
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "elapsed_sec": round(time.time() - t0, 4),
    }
    return _py(summary)


In [ ]:
%%writefile run_stage2.py
"""Write stage2_results.json from the scaled plant and a V3 learner probe.

The numeric gates are scale-conditional. ``scientific_lock`` stays false.
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

_CODEBASE = Path(__file__).resolve().parents[1] / "codebase"
if str(_CODEBASE) not in sys.path:
    sys.path.insert(0, str(_CODEBASE))

from suite import run_all  # noqa: E402


def _v3_learner_probe() -> dict:
    """A few FB updates on the vendored V3 modules. This is not an H1–H4 lock."""
    import torch

    from unit_bfm.v3.learner import Learner

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(20261024)
    learner = Learner(32, device=device, seed=20261024, config={"action_std": 0.05, "lr": 1e-4})
    n = 8
    g = torch.Generator(device="cpu").manual_seed(11)

    def rows():
        batch = {
            "x": torch.randn(n, 137, generator=g),
            "p": torch.randn(n, 32, generator=g),
            "u": torch.randn(n, 1, generator=g).tanh(),
            "x_next": torch.randn(n, 137, generator=g),
            "p_next": torch.randn(n, 32, generator=g),
            "g_next": torch.randn(n, 13, generator=g),
            "goal_g": torch.randn(n, 13, generator=g),
            "discount": torch.full((n, 1), 0.99),
            "goal_valid": torch.ones(n),
            "future_slice": (28, 32),
            "lag_slice": (24, 28),
            "config_id": torch.tensor([0, 0, 1, 1, 2, 2, 3, 3]),
        }
        batch["goal_g"][:, [0, 1, 5, 6]] = batch["x"][:, [0, 1, 5, 6]]
        batch["p"][:, 28:32] = 3.0
        batch["p"][:, 24:28] = 0.25
        batch["p_next"][:, 28:32] = 4.0
        batch["p_next"][:, 24:28] = -0.5
        return batch

    curve = []
    t0 = time.time()
    for step in range(6):
        metrics = learner.update(rows())
        curve.append({"update": step + 1, "fb": metrics["fb"], "actor": metrics["actor"]})
    return {
        "status": "executed",
        "scientific_lock": False,
        "device": device,
        "cuda_available": bool(torch.cuda.is_available()),
        "updates": learner.updates,
        "sanitize_calls": learner.sanitize_calls,
        "last_sanitize": learner.last_sanitize,
        "b_norm": float(learner.net.b(torch.randn(4, 13, device=device)).norm(dim=-1).mean().detach().cpu()),
        "curve": curve,
        "elapsed_sec": round(time.time() - t0, 4),
        "note": (
            "V3 learner path with B01 sanitizer, B02 logit hinge, and B03 "
            "matched pairs. Not a charter-scale physics run."
        ),
    }


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description="Stage-2 gate suite")
    parser.add_argument("--output", type=Path, default=Path("stage2_results.json"))
    args = parser.parse_args(argv)
    report = run_all()
    try:
        report["v3_learner_probe"] = _v3_learner_probe()
    except Exception as exc:  # noqa: BLE001 — recorded, not swallowed silently
        report["v3_learner_probe"] = {
            "status": "FAILED_TO_RUN",
            "scientific_lock": False,
            "error": f"{type(exc).__name__}: {exc}",
        }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    verdicts = report.get("metric_verdicts", {})
    print(json.dumps({"metric_verdicts": verdicts, "scientific_lock": report.get("scientific_lock"),
                      "v3_learner_probe": report["v3_learner_probe"].get("status"),
                      "output": str(args.output)}, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
import os, json
os.chdir("/content/stage2")
import run_stage2
run_stage2.main(["--output", "/content/stage2/stage2_results.json"])
report = json.load(open("/content/stage2/stage2_results.json"))
print("scientific_lock:", report["scientific_lock"])
print("metric_verdicts:", json.dumps(report["metric_verdicts"], indent=2))
print("v3_learner_probe:", report["v3_learner_probe"].get("status"), report["v3_learner_probe"].get("device"))
try:
    from google.colab import files
    files.download("/content/stage2/stage2_results.json")
except Exception as exc:
    print("download skipped:", type(exc).__name__)


## Reading the file

`metric_verdicts` records the smoke gates. H1–H4 remain Candidate in the program status even when a smoke cell says PASS. A FAIL is a failed smoke gate, not a charter lock the other way either. Re-run `run_stage2.py` locally from `programs/motor-unit-representation/stage2` after any change to the plant or the learner.
